In [1]:
# =============================================================================
# latincom_2026_FINAL.py
# Calibration without unknown data.
# Pseudo-unknown generation via (1) leave-one-family-out and
# (2) FAISS neighborhood disagreement. Real unknown used only for evaluation.
# =============================================================================

# -----------------------------------------------------------------------------
# CELL 1 — Install
# -----------------------------------------------------------------------------
!pip install -q faiss-cpu sentence-transformers transformers datasets scikit-learn \
    torch pandas numpy matplotlib seaborn tqdm psutil requests huggingface_hub \
    safetensors

import faiss, sentence_transformers, transformers
print(f"FAISS {faiss.__version__}  ST {sentence_transformers.__version__}  TF {transformers.__version__}")

# -----------------------------------------------------------------------------
# CELL 2 — Imports
# -----------------------------------------------------------------------------
import os, sys, json, time, random, gc, shutil, zipfile, pickle, warnings
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModel, ModernBertConfig, ModernBertModel)
from sklearn.metrics import (accuracy_score, f1_score, roc_curve, auc,
                             precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from google.colab import files
from tqdm import tqdm
from scipy.stats import beta as _beta, chi2 as _chi2
from safetensors.torch import load_file
from huggingface_hub import snapshot_download
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid'); sns.set_palette("husl")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 10})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=" * 80)
print("OOD-AWARE RAG-BASED NETWORK ATTACK CLASSIFICATION")
print("=" * 80)
print(f"PyTorch {torch.__version__}   CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU {torch.cuda.get_device_name(0)}")
print("=" * 80)

# -----------------------------------------------------------------------------
# CELL 3 — Download and load finetuned encoder
# -----------------------------------------------------------------------------
FINETUNED_REPO = "ccaug/modernbert-IDS-Unknown"
LOCAL_DIR = "./modernbert_ids_unknown"
os.makedirs(LOCAL_DIR, exist_ok=True)

snapshot_download(repo_id=FINETUNED_REPO, local_dir=LOCAL_DIR,
                  local_dir_use_symlinks=False, resume_download=True)
print("[download] Files:")
for f in sorted(os.listdir(LOCAL_DIR)):
    p = os.path.join(LOCAL_DIR, f)
    if os.path.isfile(p):
        print(f"   {f}  ({os.path.getsize(p)/1024/1024:.1f} MB)")

print("\nLOADING FINETUNED ENCODER")
finetuned_tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)
base_cfg = ModernBertConfig.from_pretrained("answerdotai/ModernBERT-base")
finetuned_model = ModernBertModel(base_cfg)

sd = load_file(os.path.join(LOCAL_DIR, "model.safetensors"))
mapped = {}
skipped = 0
for k, v in sd.items():
    k2 = k
    if k2.startswith("bert."):        k2 = k2[len("bert."):]
    elif k2.startswith("model."):     k2 = k2[len("model."):]
    elif k2.startswith("modernbert."):k2 = k2[len("modernbert."):]
    if any(k2.startswith(p) for p in ["classifier", "loss_fct", "head.", "temperature"]):
        skipped += 1
        continue
    mapped[k2] = v

print(f"[weights] Kept {len(mapped)}, skipped {skipped}")
missing, unexpected = finetuned_model.load_state_dict(mapped, strict=False)
print(f"[weights] missing={len(missing)}, unexpected={len(unexpected)}")
assert len(missing) == 0
finetuned_model = finetuned_model.to(DEVICE).eval()
print(f"Finetuned encoder on {DEVICE}")

print("\nLOADING BASE ENCODER")
base_tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
base_model = AutoModel.from_pretrained("answerdotai/ModernBERT-base").to(DEVICE).eval()
print("Base encoder ready")

# -----------------------------------------------------------------------------
# CELL 4 — Embedding wrapper
# -----------------------------------------------------------------------------
class EmbeddingOnlyModel:
    def __init__(self, model, tokenizer, device):
        self.model = model; self.tokenizer = tokenizer
        self.device = device; self.model.eval()
    def get_embeddings(self, texts, batch_size=32):
        embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", truncation=True,
                                     max_length=512, padding=True).to(self.device)
            with torch.no_grad():
                out = self.model(**inputs)
                last = out.last_hidden_state
                mask = inputs["attention_mask"].unsqueeze(-1)
                emb = (last * mask).sum(1) / mask.sum(1)
            emb = emb / torch.norm(emb, dim=1, keepdim=True)
            embs.append(emb.cpu().numpy())
        return np.vstack(embs)

# -----------------------------------------------------------------------------
# CELL 5 — Prototype classifier
# -----------------------------------------------------------------------------
class PrototypeClassifier:
    def __init__(self, class_prototypes):
        self.class_prototypes = class_prototypes
        self.class_names = list(class_prototypes.keys())
        self.prototype_matrix = np.stack([class_prototypes[c] for c in self.class_names])
        n = np.linalg.norm(self.prototype_matrix, axis=1, keepdims=True)
        self.prototype_matrix = self.prototype_matrix / (n + 1e-8)
    def detect_unknown(self, q):
        q = q / (np.linalg.norm(q) + 1e-8)
        s = np.dot(self.prototype_matrix, q)
        s = np.clip(s, -1.0, 1.0); s = (s + 1.0) / 2.0
        i = int(np.argmax(s))
        return self.class_names[i], float(s[i])
    def best_proto_similarity(self, q):
        q = q / (np.linalg.norm(q) + 1e-8)
        s = np.dot(self.prototype_matrix, q)
        s = np.clip(s, -1.0, 1.0); s = (s + 1.0) / 2.0
        return float(s.max())
    def predict_batch(self, embeddings):
        return [self.detect_unknown(e) for e in embeddings]

# -----------------------------------------------------------------------------
# CELL 6 — FAISS retriever with local-outlier signal
# -----------------------------------------------------------------------------
class FAISSRetriever:
    """
    Retrieves top-k neighbors and computes:
      s_ret         : mean neighbor similarity
      u_disp        : std neighbor similarity
      local_outlier : query proto sim / median neighbor self-sim  (<1 = outlier)
      disagree      : fraction of neighbors whose family != query's family
    """
    def __init__(self, index, doc_embeddings, doc_labels, pc, k=30):
        self.index = index
        self.doc_embeddings = doc_embeddings
        self.doc_labels = np.array(doc_labels)
        self.pc = pc
        self.k = k
        # Precompute each indexed doc's best prototype similarity
        self.doc_self_sim = np.array([self.pc.best_proto_similarity(e)
                                       for e in doc_embeddings])

    def retrieve_with_signals(self, q):
        q = q / (np.linalg.norm(q) + 1e-8)
        sims, idxs = self.index.search(q.reshape(1, -1).astype('float32'), self.k)
        sims = (sims[0] + 1.0) / 2.0
        idxs = idxs[0]
        valid_sims, valid_idxs = [], []
        for i, s in zip(idxs, sims):
            if 0 <= i < len(self.doc_embeddings):
                valid_sims.append(s); valid_idxs.append(i)
        if not valid_sims:
            return 0.0, 1.0, 1.0, 1.0
        v = np.array(valid_sims)
        s_ret = float(v.mean()); u_disp = float(v.std())

        neighbor_self_sims = self.doc_self_sim[valid_idxs]
        med = float(np.median(neighbor_self_sims))
        q_sim = self.pc.best_proto_similarity(q)
        local_outlier = q_sim / (med + 1e-8)

        neighbor_labels = self.doc_labels[valid_idxs]
        q_label = self.pc.detect_unknown(q)[0]
        disagree = float(np.mean(neighbor_labels != q_label))

        return s_ret, u_disp, local_outlier, disagree

# -----------------------------------------------------------------------------
# CELL 7 — Decision fusion (four-signal OR rule)
# -----------------------------------------------------------------------------
class DecisionFusion:
    def __init__(self):
        self.tau_p, self.tau_u, self.tau_r, self.tau_d = 0.5, 0.5, 1.0, 0.5
    def decide(self, s_proto, s_ret, u_disp, local_outlier, disagree):
        if s_proto < self.tau_p:       return "UNKNOWN_ATTACK"
        if u_disp > self.tau_u:        return "UNKNOWN_ATTACK"
        if local_outlier < self.tau_r: return "UNKNOWN_ATTACK"
        if disagree > self.tau_d:      return "UNKNOWN_ATTACK"
        return "KNOWN"

# -----------------------------------------------------------------------------
# CELL 8 — Build prototypes + FAISS helpers
# -----------------------------------------------------------------------------
def build_prototypes_from_embeddings(E, labels):
    protos = {}
    for cls in set(labels):
        m = np.array([l == cls for l in labels])
        ce = E[m]
        if len(ce):
            p = ce.mean(0)
            protos[cls] = p / (np.linalg.norm(p) + 1e-8)
    return protos

def build_faiss_index(E):
    e = np.asarray(E, dtype="float32")
    faiss.normalize_L2(e)
    idx = faiss.IndexFlatIP(e.shape[1])
    idx.add(e)
    return idx

# -----------------------------------------------------------------------------
# CELL 9 — Pseudo-unknown calibration (no unknown data)
# -----------------------------------------------------------------------------
class PseudoUnknownCalibrator:
    def __init__(self, E, y, k=30):
        self.E = E
        self.y = np.array(y)
        self.k = k

    def _score_sample(self, emb, pc, retr):
        _, s_proto = pc.detect_unknown(emb)
        s_ret, u_disp, lr, dg = retr.retrieve_with_signals(emb)
        return s_proto, s_ret, u_disp, lr, dg

    def _lofo(self):
        """Leave-one-family-out pseudo-labeling."""
        K, U = [], []
        families = sorted(set(self.y))
        for held in families:
            tr_mask = self.y != held
            te_mask = self.y == held
            E_tr, y_tr = self.E[tr_mask], self.y[tr_mask]
            E_te = self.E[te_mask]
            protos = build_prototypes_from_embeddings(E_tr, y_tr)
            pc = PrototypeClassifier(protos)
            idx = build_faiss_index(E_tr)
            retr = FAISSRetriever(idx, E_tr, y_tr, pc, k=self.k)
            for e in E_te:
                U.append(self._score_sample(e, pc, retr))
            for i, e in enumerate(E_tr[::3]):
                K.append(self._score_sample(e, pc, retr))
        return K, U

    def _disagreement(self):
        """Neighborhood disagreement pseudo-labeling on the full training set."""
        protos = build_prototypes_from_embeddings(self.E, self.y)
        pc = PrototypeClassifier(protos)
        idx = build_faiss_index(self.E)
        retr = FAISSRetriever(idx, self.E, self.y, pc, k=self.k)
        K, U = [], []
        for i, e in enumerate(tqdm(self.E, desc="disagreement pseudo-labels")):
            sims, idxs = idx.search(e.reshape(1, -1).astype('float32'), self.k + 1)
            idxs = idxs[0][1:]
            if len(idxs) == 0:
                continue
            labels = self.y[idxs]
            if np.all(labels == self.y[i]):
                K.append(self._score_sample(e, pc, retr))
            else:
                U.append(self._score_sample(e, pc, retr))
        return K, U

    def calibrate(self, beta=0.20):
        print("\nBuilding pseudo-unknown calibration data (no real unknown used)...")
        K1, U1 = self._lofo()
        K2, U2 = self._disagreement()
        K = np.array(K1 + K2)
        U = np.array(U1 + U2)
        print(f"   pseudo-known: {len(K)}   pseudo-unknown: {len(U)}")

        s_p_K, _, u_d_K, lr_K, dg_K = K.T
        s_p_U, _, u_d_U, lr_U, dg_U = U.T

        # Grid over the four thresholds
        gp = np.linspace(min(s_p_K.min(), s_p_U.min()), max(s_p_K.max(), s_p_U.max()), 20)
        gu = np.linspace(min(u_d_K.min(), u_d_U.min()), max(u_d_K.max(), u_d_U.max()), 20)
        gr = np.linspace(min(lr_K.min(), lr_U.min()),   max(lr_K.max(), lr_U.max()),   20)
        gd = np.linspace(min(dg_K.min(), dg_U.min()),   max(dg_K.max(), dg_U.max()),   20)

        best = None
        for tp in gp:
            for tu in gu:
                for tr in gr:
                    for td in gd:
                        rej_K = (s_p_K < tp) | (u_d_K > tu) | (lr_K < tr) | (dg_K > td)
                        frr = rej_K.mean()
                        if frr > beta: continue
                        rej_U = (s_p_U < tp) | (u_d_U > tu) | (lr_U < tr) | (dg_U > td)
                        rec = rej_U.mean()
                        if best is None or rec > best[0]:
                            best = (float(rec), float(frr),
                                    float(tp), float(tu), float(tr), float(td))

        if best is None:
            best = (0.0, 1.0,
                    float(np.percentile(s_p_K, 5)),
                    float(np.percentile(u_d_K, 95)),
                    float(np.percentile(lr_K, 5)),
                    float(np.percentile(dg_K, 95)))

        rec, frr, tp, tu, tr, td = best
        print(f"\n   Chosen thresholds:")
        print(f"      tau_p = {tp:.4f}   (proto similarity)")
        print(f"      tau_u = {tu:.4f}   (dispersion)")
        print(f"      tau_r = {tr:.4f}   (local-outlier ratio)")
        print(f"      tau_d = {td:.4f}   (neighbor disagreement)")
        print(f"   Pseudo-known FRR   : {frr:.4f}")
        print(f"   Pseudo-unknown rec : {rec:.4f}")

        return {"tau_p": tp, "tau_u": tu, "tau_r": tr, "tau_d": td,
                "pseudo_known_frr": frr, "pseudo_unknown_recall": rec,
                "n_pseudo_known": len(K), "n_pseudo_unknown": len(U)}

# -----------------------------------------------------------------------------
# CELL 10 — OOD-aware classifier
# -----------------------------------------------------------------------------
class OODAwareClassifier:
    def __init__(self, embedder_model, tokenizer, device, prototypes,
                 faiss_index, doc_embeddings, doc_labels, k=30):
        self.embedder = EmbeddingOnlyModel(embedder_model, tokenizer, device)
        self.pc = PrototypeClassifier(prototypes)
        self.retriever = FAISSRetriever(faiss_index, doc_embeddings, doc_labels,
                                         self.pc, k=k)
        self.fusion = DecisionFusion()
    def apply_thresholds(self, t):
        self.fusion.tau_p = t["tau_p"]
        self.fusion.tau_u = t["tau_u"]
        self.fusion.tau_r = t["tau_r"]
        self.fusion.tau_d = t["tau_d"]
    def classify_single(self, text):
        emb = self.embedder.get_embeddings([text])[0]
        pc, s_proto = self.pc.detect_unknown(emb)
        s_ret, u_disp, lr, dg = self.retriever.retrieve_with_signals(emb)
        final = self.fusion.decide(s_proto, s_ret, u_disp, lr, dg)
        dbg = {"proto_class": pc, "proto_score": s_proto,
               "avg_faiss_sim": s_ret, "faiss_uncertainty": u_disp,
               "local_outlier": lr, "neighbor_disagreement": dg,
               "final_class": final}
        return final, s_proto, dbg

# -----------------------------------------------------------------------------
# CELL 11 — Datasets
# -----------------------------------------------------------------------------
print("\n" + "=" * 80 + "\nDATASET UPLOAD\n" + "=" * 80)
os.makedirs("./datasets", exist_ok=True)

def ensure_file(name, required=True):
    if os.path.exists(name):
        print(f"OK  {name}"); return name
    alt = f"./datasets/{name}"
    if os.path.exists(alt):
        shutil.copy(alt, name); print(f"OK  {name}"); return name
    print(f"Upload {name}")
    up = files.upload()
    if not up:
        if required: raise RuntimeError(f"Required: {name}")
        return None
    for fn in up.keys():
        dest = f"./datasets/{fn}"
        if os.path.exists(fn): shutil.move(fn, dest)
        if not os.path.exists(name): shutil.copy(dest, name)
        print(f"   [OK] {fn}")
    return name

TRAINING_PATH = ensure_file("training_dataset.csv")
MASTER_PATH   = ensure_file("master_security_dataset.csv")
SAMPLE_PATH   = ensure_file("sample_security_dataset.csv")
TEST_PATH     = ensure_file("UNSW_NB15_testing-set.csv", required=False)

df_train  = pd.read_csv(TRAINING_PATH)
df_master = pd.read_csv(MASTER_PATH)
df_sample = pd.read_csv(SAMPLE_PATH)
df_test   = pd.read_csv(TEST_PATH) if TEST_PATH else None

print(f"\nTraining: {len(df_train)}")
print(f"Sample:   {len(df_sample)}")
print(f"Master:   {len(df_master)}")
if df_test is not None: print(f"UNSW:     {len(df_test)}")

# -----------------------------------------------------------------------------
# CELL 12 — Labels and test set
# -----------------------------------------------------------------------------
KNOWN_ATTACKS = ['DNS Fast-Flux', 'DoS', 'DoS + Brute-Force',
                 'FTP Brute-Force / Data Exfiltration', 'HTTP C2',
                 'ICMP Flood', 'IRC C2', 'P2P / UDP Scan', 'Spam']
label_mapping = {a: i for i, a in enumerate(KNOWN_ATTACKS)}
label_mapping['UNKNOWN_ATTACK'] = len(KNOWN_ATTACKS)
id_to_label = {v: k for k, v in label_mapping.items()}

df_sample['Is Known'] = df_sample['Attack Type'].apply(lambda x: x in KNOWN_ATTACKS)
df_sample_unknown = df_sample[~df_sample['Is Known']].copy()

def preprocess_unsw_row(row):
    if 'Log' in row.index: return str(row['Log'])
    parts = []
    for c in ['id','dur','proto','service','state','spkts','dpkts','sbytes',
              'dbytes','rate','sttl','dttl','sload','dload']:
        if c in row and pd.notna(row[c]):
            v = row[c]
            parts.append(f"{c}: {v:.6f}" if isinstance(v, float) else f"{c}: {v}")
    return " | ".join(parts) if parts else str(row)

UNSW_TO_KNOWN = {'DoS':'DoS','DDoS':'DoS','Fuzzers':'DoS + Brute-Force',
                 'Exploits':'HTTP C2','Generic':'Spam','Reconnaissance':'P2P / UDP Scan',
                 'Analysis':'DNS Fast-Flux','Backdoor':'IRC C2',
                 'Shellcode':'FTP Brute-Force / Data Exfiltration','Worms':'Spam',
                 'Normal':'Benign'}

test_texts, test_labels_original, test_labels = [], [], []
for attack in KNOWN_ATTACKS:
    s = df_train[df_train['Attack Type'] == attack]
    for _, row in s.sample(n=min(50, len(s)), random_state=42).iterrows():
        test_texts.append(row['Log']); test_labels_original.append(row['Attack Type'])
        test_labels.append(label_mapping[row['Attack Type']])

unk = df_sample_unknown.sample(n=min(150, len(df_sample_unknown)), random_state=42)
for _, row in unk.iterrows():
    test_texts.append(row['Log']); test_labels_original.append(row['Attack Type'])
    test_labels.append(label_mapping['UNKNOWN_ATTACK'])

if df_test is not None:
    if 'Log' not in df_test.columns:
        df_test['Log'] = df_test.apply(preprocess_unsw_row, axis=1)
    ac = next((c for c in ['attack_cat','Attack Type','label'] if c in df_test.columns), None)
    if ac:
        df_test['Mapped'] = df_test[ac].map(UNSW_TO_KNOWN).fillna('UNKNOWN_ATTACK')
        for _, row in df_test.sample(n=min(200, len(df_test)), random_state=42).iterrows():
            test_texts.append(row['Log']); ma = row['Mapped']
            test_labels_original.append(ma)
            test_labels.append(label_mapping[ma] if ma in KNOWN_ATTACKS else label_mapping['UNKNOWN_ATTACK'])

print(f"   Test set: {len(test_texts)}  known={sum(1 for l in test_labels if l != label_mapping['UNKNOWN_ATTACK'])}  unknown={sum(1 for l in test_labels if l == label_mapping['UNKNOWN_ATTACK'])}")

rag_df = df_train[df_train['Attack Type'].isin(KNOWN_ATTACKS)].copy()

# -----------------------------------------------------------------------------
# CELL 13 — Build prototypes + FAISS
# -----------------------------------------------------------------------------
print("\nBUILDING PROTOTYPES AND FAISS INDEX")
embedder = EmbeddingOnlyModel(finetuned_model, finetuned_tokenizer, DEVICE)
train_texts  = df_train['Log'].tolist()
train_labels = df_train['Attack Type'].tolist()

print("Embedding training set...")
E_train = embedder.get_embeddings(train_texts)
print(f"   Shape: {E_train.shape}")

prototypes = build_prototypes_from_embeddings(E_train, train_labels)
print(f"   Prototypes: {len(prototypes)}")

rag_texts  = rag_df['Log'].tolist()
rag_labels = rag_df['Attack Type'].tolist()
E_rag = embedder.get_embeddings(rag_texts)
idx_rag = build_faiss_index(E_rag)
print(f"   FAISS index: {idx_rag.ntotal} entries")

# -----------------------------------------------------------------------------
# CELL 14 — Calibrate without unknown data
# -----------------------------------------------------------------------------
print("\n" + "=" * 50 + "\nCALIBRATION (no unknown data)\n" + "=" * 50)
calibrator = PseudoUnknownCalibrator(E_train, train_labels, k=30)
thresholds = calibrator.calibrate(beta=0.20)

# -----------------------------------------------------------------------------
# CELL 15 — Build classifier and evaluate
# -----------------------------------------------------------------------------
print("\nEVALUATION ON REAL TEST SET")
ood_classifier = OODAwareClassifier(
    embedder_model=finetuned_model,
    tokenizer=finetuned_tokenizer,
    device=DEVICE,
    prototypes=prototypes,
    faiss_index=idx_rag,
    doc_embeddings=E_rag,
    doc_labels=rag_labels,
    k=30)
ood_classifier.apply_thresholds(thresholds)

preds, pred_ids, confs, debug_info = [], [], [], []
uid = label_mapping['UNKNOWN_ATTACK']
for text in tqdm(test_texts, desc="Classifying"):
    pc, score, dbg = ood_classifier.classify_single(text)
    pid = uid if pc == "UNKNOWN_ATTACK" else label_mapping.get(pc, uid)
    preds.append(pc); pred_ids.append(pid); confs.append(score); debug_info.append(dbg)

acc = accuracy_score(test_labels, pred_ids)
km = [l != uid for l in test_labels]; um = [l == uid for l in test_labels]
ka = accuracy_score([test_labels[i] for i in range(len(test_labels)) if km[i]],
                    [pred_ids[i] for i in range(len(pred_ids)) if km[i]]) if any(km) else 0
ua = accuracy_score([test_labels[i] for i in range(len(test_labels)) if um[i]],
                    [pred_ids[i] for i in range(len(pred_ids)) if um[i]]) if any(um) else 0
f1m = f1_score(test_labels, pred_ids, average='macro', zero_division=0)

print(f"\nFINAL RESULTS (calibrated without unknown data)")
print(f"   Overall accuracy : {acc:.4f}")
print(f"   Known accuracy   : {ka:.4f}")
print(f"   Unknown accuracy : {ua:.4f}")
print(f"   F1 macro         : {f1m:.4f}")

# Detailed diagnostic
pred_unknown = np.array([1 if p == "UNKNOWN_ATTACK" else 0 for p in preds])
km_ = np.array(km); um_ = np.array(um)
known_acc_native = float((1 - pred_unknown[km_]).mean())
frr_native = float(pred_unknown[km_].mean())
unk_recall_native = float(pred_unknown[um_].mean())
far_native = 1 - unk_recall_native
n_u = int(um_.sum()); x_u = int((~pred_unknown[um_]).sum())
cp_up = float(_beta.ppf(0.95, x_u+1, n_u-x_u)) if x_u < n_u else 1.0

print(f"\nNative decisions:")
print(f"   Known accuracy    : {known_acc_native:.4f}")
print(f"   Unknown recall    : {unk_recall_native:.4f}")
print(f"   FAR               : {far_native:.4f}")
print(f"   FAR 95% CP upper  : {cp_up:.4f}")
print(f"   FRR               : {frr_native:.4f}")

# -----------------------------------------------------------------------------
# CELL 16 — Per-family breakdown
# -----------------------------------------------------------------------------
print("\nPER-FAMILY RESULTS")
per_fam = []
for fam in sorted(set(test_labels_original)):
    idxs = [i for i, t in enumerate(test_labels_original) if t == fam]
    if not idxs: continue
    fam_pred = pred_unknown[idxs]
    is_unk = (label_mapping.get(fam, uid) == uid)
    if is_unk:
        rec = float(fam_pred.mean())
        per_fam.append({"Family": fam, "Type": "unknown", "N": len(idxs),
                        "Unknown Rec.": rec, "FAR": 1 - rec})
    else:
        acc_ = float((1 - fam_pred).mean())
        per_fam.append({"Family": fam, "Type": "known", "N": len(idxs),
                        "Known Acc": acc_, "FRR": float(fam_pred.mean())})

for r in per_fam:
    if r["Type"] == "unknown":
        print(f"   UNK {r['Family']:35s} N={r['N']:3d}  recall={r['Unknown Rec.']:.4f}")
    else:
        print(f"   KNW {r['Family']:35s} N={r['N']:3d}  acc   ={r['Known Acc']:.4f}")

# -----------------------------------------------------------------------------
# CELL 17 — Baselines on the same test set
# -----------------------------------------------------------------------------
print("\nBENCHMARK BASELINES")
df_sig = pd.DataFrame({
    'proto_score':        [d['proto_score'] for d in debug_info],
    'avg_faiss_sim':      [d['avg_faiss_sim'] for d in debug_info],
    'faiss_uncertainty':  [d['faiss_uncertainty'] for d in debug_info],
    'local_outlier':      [d['local_outlier'] for d in debug_info],
    'neighbor_disagree':  [d['neighbor_disagreement'] for d in debug_info]})

y_true = np.array([1 if l != uid else 0 for l in test_labels])
U_MAX = float(df_sig['faiss_uncertainty'].max()) + 1e-9
df_sig['score_fused'] = np.minimum(
    np.minimum(df_sig['proto_score'], df_sig['avg_faiss_sim']),
    1.0 - df_sig['faiss_uncertainty'] / U_MAX)
df_sig['score_proto'] = df_sig['proto_score']

BETA = 0.20
def threshold_under_frr(sk, su, beta=BETA):
    lo = float(min(sk.min(), su.min())); hi = float(max(sk.max(), su.max()))
    best = None
    for tau in np.linspace(lo, hi, 400):
        frr = (sk < tau).mean(); rec = (su < tau).mean()
        if frr > beta: continue
        if best is None or rec > best[0]: best = (float(rec), float(frr), float(tau))
    if best is None: best = (0.0, 1.0, hi)
    return best

def evaluate_score(name, scores, y):
    scores = np.asarray(scores, dtype=float)
    sk = scores[y == 1]; su = scores[y == 0]
    rec, frr, tau = threshold_under_frr(sk, su, BETA)
    fpr, tpr, _ = roc_curve(y, scores)
    return {"Method": name, "Unknown Rec.": rec, "FAR": 1-rec,
            "FRR": frr, "AUROC": float(auc(fpr, tpr))}

E_train_b = E_train
y_train_b = np.array([label_mapping[t] for t in train_labels])
E_test_b  = embedder.get_embeddings(test_texts)
E_rag_b   = E_rag

head = LogisticRegression(max_iter=2000, n_jobs=-1).fit(E_train_b, y_train_b)
msp    = head.predict_proba(E_test_b).max(1)
energy = head.decision_function(E_test_b).max(1)
mus, precs = {}, {}
for c in np.unique(y_train_b):
    Xc = E_train_b[y_train_b == c]
    mus[c] = Xc.mean(0)
    cov = np.cov(Xc.T) + 1e-3 * np.eye(E_train_b.shape[1])
    precs[c] = np.linalg.inv(cov)
D = np.stack([((E_test_b - mus[c]) @ precs[c] * (E_test_b - mus[c])).sum(1)
              for c in mus], 1)
maha = -D.min(1)

idx_tr = faiss.IndexFlatIP(E_train_b.shape[1]); idx_tr.add(E_train_b.astype('float32'))
D_knn, _ = idx_tr.search(E_test_b.astype('float32'), 30); knn = D_knn.mean(1)

idx_rag_b = faiss.IndexFlatIP(E_rag_b.shape[1]); idx_rag_b.add(E_rag_b.astype('float32'))
D_r, I_r = idx_rag_b.search(E_test_b.astype('float32'), 10)
rag_lbl = rag_df['Attack Type'].values
vote = np.array([pd.Series(rag_lbl[row]).value_counts().iloc[0] / 10 for row in I_r])

soft = head.predict_proba(E_test_b).clip(1e-9, 1)
soft = soft ** (1/1.5); soft = soft / soft.sum(1, keepdims=True); soft = soft.max(1)

rows = []
for name, scores in [
    ('MSP', msp), ('Energy', energy), ('Mahalanobis', maha),
    ('kNN-OOD', knn), ('Retrieval vote', vote),
    ('Softmax + temperature', soft),
    ('Prototype only', df_sig['score_proto'].values),
    ('Proposed (four-signal OR)', pred_unknown.astype(float)),
]:
    rows.append(evaluate_score(name, scores, y_true))

df_bench = pd.DataFrame(rows).sort_values('Unknown Rec.', ascending=False)
print("\nBenchmark summary")
print("-" * 80)
print(f"{'Method':35s} {'Unknown Rec.':>12s} {'FAR':>8s} {'FRR':>8s} {'AUROC':>8s}")
print("-" * 80)
for _, r in df_bench.iterrows():
    print(f"{r['Method']:35s} {r['Unknown Rec.']:12.4f} {r['FAR']:8.4f} {r['FRR']:8.4f} {r['AUROC']:8.4f}")
print("-" * 80)

# -----------------------------------------------------------------------------
# CELL 18 — Save + summary
# -----------------------------------------------------------------------------
summary = {
    "calibration": thresholds,
    "native": {
        "known_accuracy": known_acc_native,
        "unknown_recall": unk_recall_native,
        "FAR": far_native,
        "FAR_95pct_cp_upper": cp_up,
        "FRR": frr_native,
    },
    "benchmark": df_bench.to_dict(orient="records"),
    "per_family": per_fam,
}
with open("benchmark_results.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"Calibration used NO real unknown data.")
print(f"Pseudo-unknown sources: leave-one-family-out + FAISS neighborhood disagreement.")
print(f"  pseudo-known FRR   = {thresholds['pseudo_known_frr']:.4f}")
print(f"  pseudo-unknown rec = {thresholds['pseudo_unknown_recall']:.4f}")
print(f"")
print(f"Real test set performance:")
print(f"  Known accuracy   = {known_acc_native:.4f}")
print(f"  Unknown recall   = {unk_recall_native:.4f}")
print(f"  FAR              = {far_native:.4f}")
print(f"  FRR              = {frr_native:.4f}")
print("=" * 80)

zip_path = "benchmark_results.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write("benchmark_results.json")
print(f"[saved] {zip_path}")
try:
    files.download(zip_path)
except Exception:
    pass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.3 MB/s eta 0:00:00
FAISS 1.15.1  ST 5.7.0  TF 5.16.1
OOD-AWARE RAG-BASED NETWORK ATTACK CLASSIFICATION
PyTorch 2.11.0+cu128   CUDA True
GPU Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[download] Files:
   .gitattributes  (0.0 MB)
   config.json  (0.0 MB)
   model.safetensors  (569.9 MB)
   tokenizer.json  (3.4 MB)
   tokenizer_config.json  (0.0 MB)
   training_args.bin  (0.0 MB)

LOADING FINETUNED ENCODER


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

...

   pseudo-known: 16148   pseudo-unknown: 4858

   Chosen thresholds:
      tau_p = 0.9772   (proto similarity)
      tau_u = 0.0043   (dispersion)
      tau_r = 0.9909   (local-outlier ratio)
      tau_d = 0.0000   (neighbor disagreement)
   Pseudo-known FRR   : 0.1168
   Pseudo-unknown rec : 0.9981

EVALUATION ON REAL TEST SET


Classifying: 100%|██████████| 800/800 [00:17<00:00, 45.25it/s]



FINAL RESULTS (calibrated without unknown data)
   Overall accuracy : 0.3100
   Known accuracy   : 0.0000
   Unknown accuracy : 1.0000
   F1 macro         : 0.0473

Native decisions:
   Known accuracy    : 0.6902
   Unknown recall    : 1.0000
   FAR               : 0.0000
   FAR 95% CP upper  : nan
   FRR               : 0.3098

PER-FAMILY RESULTS
   UNK Anomaly                             N= 29  recall=1.0000
   UNK Benign                              N=121  recall=1.0000
   UNK Bot                                 N= 24  recall=1.0000
   UNK Brute_Force                         N= 26  recall=1.0000
   UNK DDoS                                N= 15  recall=1.0000
   KNW DNS Fast-Flux                       N= 53  acc   =0.6415
   KNW DoS                                 N= 58  acc   =0.6552
   KNW DoS + Brute-Force                   N= 60  acc   =0.8333
   UNK Dos Attacks-Goldeneye               N= 18  recall=1.0000
   KNW FTP Brute-Force / Data Exfiltration N= 51  acc   =0.8627
   KNW HT

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# =============================================================================
# CELL 18 — Corrected calibrator: two signals (proto + dispersion),
# tie-broken by pseudo-known FRR. Fixes the collapse to tau_d=0.0.
# =============================================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import beta as _beta
import json, zipfile

class PseudoUnknownCalibratorV2:
    """
    Two-signal calibration:
      - s_proto   (prototype similarity)
      - u_disp    (retrieval dispersion)

    For each beta, choose the (tau_p, tau_u) tuple that
      (a) satisfies pseudo-known FRR <= beta AND held-out known FRR <= beta,
      (b) maximizes pseudo-unknown recall,
      (c) among ties, minimizes pseudo-known FRR.
    The tie-break on (c) prevents the search from picking a corner of the
    grid that fires on every sample.
    """
    def __init__(self, E, y, k=30):
        self.E = E
        self.y = np.array(y)
        self.k = k
        # Cache the pseudo distributions once.
        print("Precomputing pseudo-known and pseudo-unknown distributions...")
        K1, U1 = self._lofo()
        K2, U2 = self._disagreement()
        self.K = np.array(K1 + K2)
        self.U = np.array(U1 + U2)
        self.V = self._held_out_known_validation()
        print(f"   pseudo-known: {len(self.K)}   pseudo-unknown: {len(self.U)}   "
              f"held-out known: {len(self.V)}")

    def _score_sample(self, emb, pc, retr):
        _, s_proto = pc.detect_unknown(emb)
        _, u_disp, _, _ = retr.retrieve_with_signals(emb)
        return s_proto, u_disp

    def _lofo(self):
        K, U = [], []
        for held in sorted(set(self.y)):
            tr_mask = self.y != held
            te_mask = self.y == held
            E_tr, y_tr = self.E[tr_mask], self.y[tr_mask]
            E_te = self.E[te_mask]
            protos = build_prototypes_from_embeddings(E_tr, y_tr)
            pc = PrototypeClassifier(protos)
            idx = build_faiss_index(E_tr)
            retr = FAISSRetriever(idx, E_tr, y_tr, pc, k=self.k)
            for e in E_te:
                U.append(self._score_sample(e, pc, retr))
            for e in E_tr[::3]:
                K.append(self._score_sample(e, pc, retr))
        return K, U

    def _disagreement(self):
        protos = build_prototypes_from_embeddings(self.E, self.y)
        pc = PrototypeClassifier(protos)
        idx = build_faiss_index(self.E)
        retr = FAISSRetriever(idx, self.E, self.y, pc, k=self.k)
        K, U = [], []
        for i, e in enumerate(tqdm(self.E, desc="disagreement pseudo-labels")):
            sims, idxs = idx.search(e.reshape(1, -1).astype('float32'), self.k + 1)
            idxs = idxs[0][1:]
            if len(idxs) == 0:
                continue
            labels = self.y[idxs]
            if np.all(labels == self.y[i]):
                K.append(self._score_sample(e, pc, retr))
            else:
                U.append(self._score_sample(e, pc, retr))
        return K, U

    def _held_out_known_validation(self, seed=1, frac=0.15):
        rng = np.random.RandomState(seed)
        n = len(self.E)
        val_idx = rng.permutation(n)[:int(n * frac)]
        E_val = self.E[val_idx]
        protos = build_prototypes_from_embeddings(self.E, self.y)
        pc = PrototypeClassifier(protos)
        idx = build_faiss_index(self.E)
        retr = FAISSRetriever(idx, self.E, self.y, pc, k=self.k)
        return np.array([self._score_sample(e, pc, retr) for e in E_val])

    def calibrate(self, beta=0.15):
        s_p_K, u_d_K = self.K.T
        s_p_U, u_d_U = self.U.T
        s_p_V, u_d_V = self.V.T

        # Denser grid, restricted to the top quantiles of the known
        # distributions so the search is not tempted by degenerate corners.
        gp = np.linspace(np.percentile(s_p_K, 50), np.percentile(s_p_K, 99), 25)
        gu = np.linspace(np.percentile(u_d_K, 1),  np.percentile(u_d_K, 99), 25)

        best = None
        for tp in gp:
            for tu in gu:
                rej_V = (s_p_V < tp) | (u_d_V > tu)
                if rej_V.mean() > beta: continue
                rej_K = (s_p_K < tp) | (u_d_K > tu)
                frr = rej_K.mean()
                if frr > beta: continue
                rej_U = (s_p_U < tp) | (u_d_U > tu)
                rec = rej_U.mean()
                # Primary: recall. Secondary: lower FRR. Tertiary: higher tau_p.
                key = (round(rec, 4), -round(frr, 4), round(tp, 4))
                if best is None or key > best[0]:
                    best = (key, rec, frr, tp, tu)

        if best is None:
            tp = float(np.percentile(s_p_K, 5))
            tu = float(np.percentile(u_d_K, 95))
            return {"tau_p": tp, "tau_u": tu,
                    "pseudo_known_frr": 1.0, "pseudo_unknown_recall": 0.0}

        _, rec, frr, tp, tu = best
        print(f"   beta={beta:.2f}  tau_p={tp:.4f}  tau_u={tu:.4f}  "
              f"pseudo_FRR={frr:.4f}  pseudo_rec={rec:.4f}")
        return {"tau_p": float(tp), "tau_u": float(tu),
                "pseudo_known_frr": float(frr),
                "pseudo_unknown_recall": float(rec)}

# ---------------------------------------------------------------------------
# Build the calibrator once (precomputes distributions) and then sweep.
# ---------------------------------------------------------------------------
print("=" * 80)
print("CORRECTED CALIBRATION (two signals, tie-broken by FRR)")
print("=" * 80)

cal_v2 = PseudoUnknownCalibratorV2(E_train, train_labels, k=30)

print("\n" + "=" * 80)
print("BUDGET SWEEP")
print("=" * 80)

sweep_rows = []
uid = label_mapping["UNKNOWN_ATTACK"]
km = np.array([l != uid for l in test_labels])
um = np.array([l == uid for l in test_labels])

# Snapshot the original thresholds so we can restore them at the end.
orig = {
    "tau_p": ood_classifier.fusion.tau_p,
    "tau_u": ood_classifier.fusion.tau_u,
    "tau_r": ood_classifier.fusion.tau_r,
    "tau_d": ood_classifier.fusion.tau_d,
}

for beta_val in [0.20, 0.18, 0.15, 0.12, 0.10, 0.08, 0.06, 0.05]:
    t = cal_v2.calibrate(beta=beta_val)

    # Apply the two-signal thresholds. Keep tau_r and tau_d effectively
    # disabled by setting them to values that never fire.
    ood_classifier.fusion.tau_p = t["tau_p"]
    ood_classifier.fusion.tau_u = t["tau_u"]
    ood_classifier.fusion.tau_r = -1.0     # never fires (local_outlier always >= 0)
    ood_classifier.fusion.tau_d = 1.0 + 1e-9  # never fires (disagreement always <= 1)

    preds_b = []
    for text in test_texts:
        pc, _, _ = ood_classifier.classify_single(text)
        preds_b.append(pc)

    pred_u_b = np.array([1 if p == "UNKNOWN_ATTACK" else 0 for p in preds_b])
    ka_b = float((1 - pred_u_b[km]).mean())
    ua_b = float(pred_u_b[um].mean())
    fr_b = float(pred_u_b[km].mean())

    sweep_rows.append({
        "beta": beta_val,
        "known_acc": ka_b,
        "unknown_recall": ua_b,
        "FAR": 1 - ua_b,
        "FRR": fr_b,
        "tau_p": t["tau_p"], "tau_u": t["tau_u"],
    })
    print(f"   beta={beta_val:.2f}  tau_p={t['tau_p']:.4f}  tau_u={t['tau_u']:.4f}  "
          f"known_acc={ka_b:.4f}  unknown_recall={ua_b:.4f}  "
          f"FAR={1-ua_b:.4f}  FRR={fr_b:.4f}")

df_sweep = pd.DataFrame(sweep_rows)
print("\nBudget sweep summary:")
print(df_sweep.to_string(index=False))

# ---------------------------------------------------------------------------
# Restore the beta=0.15 operating point.
# ---------------------------------------------------------------------------
row15 = df_sweep[df_sweep["beta"] == 0.15]
if not row15.empty:
    r = row15.iloc[0]
    ood_classifier.fusion.tau_p = float(r["tau_p"])
    ood_classifier.fusion.tau_u = float(r["tau_u"])
    ood_classifier.fusion.tau_r = -1.0
    ood_classifier.fusion.tau_d = 1.0 + 1e-9
    print(f"\nRestored beta=0.15 operating point: "
          f"tau_p={r['tau_p']:.4f}  tau_u={r['tau_u']:.4f}")

    # Re-evaluate at the restored operating point.
    preds_final = []
    for text in test_texts:
        pc, _, _ = ood_classifier.classify_single(text)
        preds_final.append(pc)
    pred_u_final = np.array([1 if p == "UNKNOWN_ATTACK" else 0 for p in preds_final])
    ka_final = float((1 - pred_u_final[km]).mean())
    ua_final = float(pred_u_final[um].mean())
    fr_final = float(pred_u_final[km].mean())

    print("\n" + "=" * 80)
    print("FINAL AT beta=0.15")
    print("=" * 80)
    print(f"   Known accuracy    : {ka_final:.4f}")
    print(f"   Unknown recall    : {ua_final:.4f}")
    print(f"   FAR               : {1-ua_final:.4f}")
    print(f"   FRR               : {fr_final:.4f}")
    print("=" * 80)

    # Save.
    out = {
        "budget_sweep": sweep_rows,
        "final_beta_0.15": {
            "known_accuracy": ka_final,
            "unknown_recall": ua_final,
            "FAR": 1 - ua_final,
            "FRR": fr_final,
            "tau_p": float(r["tau_p"]),
            "tau_u": float(r["tau_u"]),
        },
    }
    with open("benchmark_results_v3.json", "w") as f:
        json.dump(out, f, indent=2, default=str)
    print("[saved] benchmark_results_v3.json")

    try:
        from google.colab import files as _f
        _f.download("benchmark_results_v3.json")
        print("[download] started")
    except Exception as e:
        print(f"[note] {e}")

CORRECTED CALIBRATION (two signals, tie-broken by FRR)
Precomputing pseudo-known and pseudo-unknown distributions...


disagreement pseudo-labels: 100%|██████████| 4500/4500 [00:13<00:00, 345.13it/s]


   pseudo-known: 16148   pseudo-unknown: 4858   held-out known: 675

BUDGET SWEEP
   beta=0.20  tau_p=0.9712  tau_u=0.0064  known_acc=0.7645  unknown_recall=1.0000  FAR=0.0000  FRR=0.2355
   beta=0.18  tau_p=0.9712  tau_u=0.0064  known_acc=0.7645  unknown_recall=1.0000  FAR=0.0000  FRR=0.2355
   beta=0.15  tau_p=0.9712  tau_u=0.0064  known_acc=0.7645  unknown_recall=1.0000  FAR=0.0000  FRR=0.2355
   beta=0.12  tau_p=0.9712  tau_u=0.0064  known_acc=0.7645  unknown_recall=1.0000  FAR=0.0000  FRR=0.2355
   beta=0.10  tau_p=0.9712  tau_u=0.0064  known_acc=0.7645  unknown_recall=1.0000  FAR=0.0000  FRR=0.2355
   beta=0.08  tau_p=0.9712  tau_u=0.0064  known_acc=0.7645  unknown_recall=1.0000  FAR=0.0000  FRR=0.2355
   beta=0.06  tau_p=0.9712  tau_u=0.0064  known_acc=0.7645  unknown_recall=1.0000  FAR=0.0000  FRR=0.2355
   beta=0.05  tau_p=0.9712  tau_u=0.0064  known_acc=0.7645  unknown_recall=1.0000  FAR=0.0000  FRR=0.2355

Budget sweep summary:
 beta  known_acc  unknown_recall  FAR      FRR 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[download] started


In [3]:
# =============================================================================
# CELL 20 — RAG corpus scaling benchmark + model comparison
# =============================================================================
# Two experiments in one cell:
#
# EXP 1 — RAG corpus scaling:
#   Start with 500 training samples in the FAISS index. Move 500 samples
#   per step from the RAG index into the "held-out" set, which joins the
#   evaluation pool. The remaining 500 stay in the index. At each step,
#   re-evaluate the same classifier on a growing test set built as:
#       test_set(step k) = original_test_set ∪ (training samples removed from RAG)
#   This shows how the classifier's decisions and the calibration depend on
#   corpus size, and it also gives a growing evaluation set.
#
# EXP 2 — Model comparison at fixed corpus sizes:
#   Same protocol, but repeat the evaluation with several candidate models
#   on the same splits:
#     - Proposed (ModernBERT finetuned + prototypes + dispersion + pseudo-unknown calibration)
#     - ModernBERT finetuned + prototype-only rejection (single-signal ablation)
#     - ModernBERT finetuned + kNN-OOD rejection
#     - ModernBERT base + prototype rejection
#     - Random Forest on TF-IDF with softmax confidence rejection
#     - kNN-OOD on ModernBERT base embeddings
#   All models share the same evaluation set at every step, so the comparison
#   is on identical data.
#
# Uses objects already in memory: E_train, train_labels, embedder,
# finetuned_model, finetuned_tokenizer, base_model, base_tokenizer,
# test_texts, test_labels, test_labels_original, label_mapping, id_to_label.
# =============================================================================

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import beta as _beta
import faiss, json, zipfile, os

# ---------------------------------------------------------------------------
# Helpers we need (keep self-contained in case a name went out of scope)
# ---------------------------------------------------------------------------
def _build_protos(E, y):
    protos = {}
    for cls in set(y):
        m = np.array([l == cls for l in y])
        ce = E[m]
        if len(ce):
            p = ce.mean(0)
            protos[cls] = p / (np.linalg.norm(p) + 1e-8)
    return protos

def _faiss_index(E):
    e = np.asarray(E, dtype="float32")
    faiss.normalize_L2(e)
    idx = faiss.IndexFlatIP(e.shape[1])
    idx.add(e)
    return idx

def _score_proto_disp(q, pc, idx, doc_self_sims):
    """Return (s_proto, u_disp) for a single query embedding."""
    qq = q / (np.linalg.norm(q) + 1e-8)
    # prototype similarity
    sims = np.dot(pc.prototype_matrix, qq)
    sims = np.clip(sims, -1.0, 1.0)
    sims = (sims + 1.0) / 2.0
    s_proto = float(sims.max())
    # retrieval dispersion
    D, _ = idx.search(qq.reshape(1, -1).astype("float32"), 30)
    D = (D[0] + 1.0) / 2.0
    u_disp = float(D.std())
    return s_proto, u_disp

# ---------------------------------------------------------------------------
# Set up the base test set (constant across all steps)
# ---------------------------------------------------------------------------
uid = label_mapping["UNKNOWN_ATTACK"]
TEST_BASE_TEXTS   = list(test_texts)
TEST_BASE_LABELS  = list(test_labels)
TEST_BASE_ORIGINAL = list(test_labels_original)

# All training samples that will be progressively pulled out of the RAG index.
# Preserve the order so the splits are deterministic.
TRAIN_FULL_TEXTS  = list(df_train["Log"].tolist())
TRAIN_FULL_LABELS = list(df_train["Attack Type"].tolist())

# ---------------------------------------------------------------------------
# Evaluation helpers
# ---------------------------------------------------------------------------
def metrics_from_predictions(labels, pred_unknown_flag, pred_classes=None):
    labels = np.asarray(labels)
    pred_unknown_flag = np.asarray(pred_unknown_flag)
    km = (labels != uid)
    um = (labels == uid)
    known_acc = float((1 - pred_unknown_flag[km]).mean()) if km.any() else float("nan")
    unk_rec   = float(pred_unknown_flag[um].mean()) if um.any() else float("nan")
    far       = float(1 - unk_rec) if not np.isnan(unk_rec) else float("nan")
    frr       = float(pred_unknown_flag[km].mean()) if km.any() else float("nan")
    return {
        "known_accuracy": known_acc,
        "unknown_recall": unk_rec,
        "FAR": far,
        "FRR": frr,
        "n_known": int(km.sum()),
        "n_unknown": int(um.sum()),
    }

# ---------------------------------------------------------------------------
# Proposed pipeline: calibrate a two-signal (proto + dispersion) rule using
# pseudo-unknown distributions from LOFO + disagreement on the CURRENT
# retrieval corpus. This uses no real unknown data.
# ---------------------------------------------------------------------------
def calibrate_two_signal(E_corpus, y_corpus, beta=0.15, k=30):
    """Return (tau_p, tau_u). Uses leave-one-family-out on the current corpus."""
    if len(set(y_corpus)) < 2:
        # Cannot hold out families; fall back to a permissive threshold.
        return float(np.percentile([
            _score_proto_disp(e, PrototypeClassifier(_build_protos(E_corpus, y_corpus)),
                              _faiss_index(E_corpus), None)[0]
            for e in E_corpus[:100]
        ], 5)), 1.0

    K, U = [], []
    for held in sorted(set(y_corpus)):
        tr_mask = np.array([y == held for y in y_corpus]) == False
        te_mask = np.array([y == held for y in y_corpus])
        if tr_mask.sum() < 5 or te_mask.sum() < 1:
            continue
        E_tr = E_corpus[tr_mask]; y_tr = np.array(y_corpus)[tr_mask]
        E_te = E_corpus[te_mask]
        protos = _build_protos(E_tr, y_tr)
        pc = PrototypeClassifier(protos)
        idx = _faiss_index(E_tr)
        for e in E_te[:100]:
            K.append(_score_proto_disp(e, pc, idx, None))   # placeholder for known
            K.pop()  # we don't record held family as known
        # Held-out family = pseudo-unknown
        for e in E_te[:200]:
            U.append(_score_proto_disp(e, pc, idx, None))
        # Remaining families = pseudo-known
        for i, e in enumerate(E_tr[::5]):
            K.append(_score_proto_disp(e, pc, idx, None))

    if len(K) == 0 or len(U) == 0:
        return 0.5, 0.5

    K = np.array(K); U = np.array(U)
    s_p_K, u_d_K = K.T
    s_p_U, u_d_U = U.T

    gp = np.linspace(np.percentile(s_p_K, 50), np.percentile(s_p_K, 99), 20)
    gu = np.linspace(np.percentile(u_d_K, 1),  np.percentile(u_d_K, 99), 20)

    best = None
    for tp in gp:
        for tu in gu:
            frr = ((s_p_K < tp) | (u_d_K > tu)).mean()
            if frr > beta: continue
            rec = ((s_p_U < tp) | (u_d_U > tu)).mean()
            key = (round(rec, 4), -round(frr, 4), round(tp, 4))
            if best is None or key > best[0]:
                best = (key, tp, tu)
    if best is None:
        return float(np.percentile(s_p_K, 5)), float(np.percentile(u_d_K, 95))
    return float(best[1]), float(best[2])

def evaluate_proposed(E_corpus, y_corpus, texts_eval, labels_eval,
                      beta=0.15, label="Proposed"):
    tau_p, tau_u = calibrate_two_signal(E_corpus, y_corpus, beta=beta)
    protos = _build_protos(E_corpus, y_corpus)
    pc = PrototypeClassifier(protos)
    idx = _faiss_index(E_corpus)
    flags = []
    classes = []
    for text in tqdm(texts_eval, desc=f"{label} eval", leave=False):
        q = embedder.get_embeddings([text])[0]
        s_proto, u_disp = _score_proto_disp(q, pc, idx, None)
        if s_proto < tau_p or u_disp > tau_u:
            flags.append(1); classes.append("UNKNOWN_ATTACK")
        else:
            cls, _ = pc.detect_unknown(q)
            flags.append(0); classes.append(cls)
    m = metrics_from_predictions(labels_eval, flags, classes)
    m["tau_p"] = tau_p; m["tau_u"] = tau_u
    return m

# ---------------------------------------------------------------------------
# Prototype-only ablation
# ---------------------------------------------------------------------------
def evaluate_prototype_only(E_corpus, y_corpus, texts_eval, labels_eval,
                             beta=0.15, label="Prototype-only"):
    # Calibrate a single threshold on pseudo-unknown prototype similarity only.
    K, U = [], []
    for held in sorted(set(y_corpus)):
        tr_mask = np.array([y == held for y in y_corpus]) == False
        te_mask = np.array([y == held for y in y_corpus])
        if tr_mask.sum() < 5 or te_mask.sum() < 1:
            continue
        E_tr = E_corpus[tr_mask]; y_tr = np.array(y_corpus)[tr_mask]
        E_te = E_corpus[te_mask]
        protos = _build_protos(E_tr, y_tr)
        pc = PrototypeClassifier(protos)
        for e in E_te[:200]:
            s, _ = _score_proto_disp(e, pc, _faiss_index(E_tr), None)
            U.append(s)
        for e in E_tr[::5]:
            s, _ = _score_proto_disp(e, pc, _faiss_index(E_tr), None)
            K.append(s)
    if len(K) == 0 or len(U) == 0:
        tau_p = 0.5
    else:
        K = np.array(K); U = np.array(U)
        gp = np.linspace(np.percentile(K, 50), np.percentile(K, 99), 25)
        best = None
        for tp in gp:
            frr = (K < tp).mean()
            if frr > beta: continue
            rec = (U < tp).mean()
            key = (round(rec, 4), -round(frr, 4), round(tp, 4))
            if best is None or key > best[0]:
                best = (key, tp)
        tau_p = best[1] if best else float(np.percentile(K, 5))

    protos = _build_protos(E_corpus, y_corpus)
    pc = PrototypeClassifier(protos)
    flags, classes = [], []
    for text in tqdm(texts_eval, desc=f"{label} eval", leave=False):
        q = embedder.get_embeddings([text])[0]
        s_proto, _ = _score_proto_disp(q, pc, _faiss_index(E_corpus), None)
        if s_proto < tau_p:
            flags.append(1); classes.append("UNKNOWN_ATTACK")
        else:
            cls, _ = pc.detect_unknown(q)
            flags.append(0); classes.append(cls)
    m = metrics_from_predictions(labels_eval, flags, classes)
    m["tau_p"] = tau_p
    return m

# ---------------------------------------------------------------------------
# kNN-OOD baseline
# ---------------------------------------------------------------------------
def evaluate_knn_ood(E_corpus, y_corpus, texts_eval, labels_eval,
                     beta=0.15, label="kNN-OOD"):
    idx = _faiss_index(E_corpus)
    # Calibrate the kNN similarity threshold on known-only data by holding
    # out a random 20% of the corpus and using the 15th percentile as tau.
    rng = np.random.RandomState(0)
    n = len(E_corpus)
    holdout = rng.permutation(n)[:max(50, int(0.2 * n))]
    E_hold = E_corpus[holdout]
    idx_train = _faiss_index(E_corpus)
    sims = []
    for e in E_hold:
        D, _ = idx_train.search(e.reshape(1, -1).astype("float32"), 30)
        sims.append((D[0] + 1.0).mean() / 2.0)
    tau = float(np.percentile(sims, beta * 100))

    flags, classes = [], []
    for text in tqdm(texts_eval, desc=f"{label} eval", leave=False):
        q = embedder.get_embeddings([text])[0]
        D, _ = idx.search(q.reshape(1, -1).astype("float32"), 30)
        s = float(((D[0] + 1.0) / 2.0).mean())
        if s < tau:
            flags.append(1); classes.append("UNKNOWN_ATTACK")
        else:
            protos = _build_protos(E_corpus, y_corpus)
            pc = PrototypeClassifier(protos)
            cls, _ = pc.detect_unknown(q)
            flags.append(0); classes.append(cls)
    m = metrics_from_predictions(labels_eval, flags, classes)
    m["tau"] = tau
    return m

# ---------------------------------------------------------------------------
# Base ModernBERT + prototype rejection
# ---------------------------------------------------------------------------
base_embedder = EmbeddingOnlyModel(base_model, base_tokenizer, DEVICE)
def evaluate_base_proto(E_corpus_ft, y_corpus, texts_eval, labels_eval,
                         beta=0.15, label="Base-proto"):
    # Re-embed the corpus with the BASE encoder
    E_corpus_base = base_embedder.get_embeddings(
        [df_train["Log"].tolist()[i] for i in range(len(E_corpus_ft))][:len(E_corpus_ft)]
    ) if False else None
    # simpler: the caller passes the corpus texts; we re-embed here.
    return None  # handled in main loop

# ---------------------------------------------------------------------------
# Random Forest closed-set with a calibrated softmax rejection
# ---------------------------------------------------------------------------
def evaluate_rf(texts_corpus, y_corpus, texts_eval, labels_eval,
                beta=0.15, label="RandomForest"):
    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    Xc = tfidf.fit_transform(texts_corpus)
    yc = np.array([label_mapping.get(t, uid) for t in y_corpus])
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(Xc, yc)
    Xe = tfidf.transform(texts_eval)
    probs = rf.predict_proba(Xe)
    conf = probs.max(1)
    tau = float(np.percentile(conf, beta * 100))
    flags = (conf < tau).astype(int)
    classes = []
    for i, f in enumerate(flags):
        if f:
            classes.append("UNKNOWN_ATTACK")
        else:
            classes.append(id_to_label[int(np.argmax(probs[i]))])
    m = metrics_from_predictions(labels_eval, flags, classes)
    m["tau"] = tau
    return m

# ---------------------------------------------------------------------------
# MAIN: progressive RAG reduction
# ---------------------------------------------------------------------------
print("=" * 80)
print("RAG CORPUS SCALING — 500 to 4500 in steps of 500")
print("=" * 80)

# We start with the smallest corpus (500 samples) and progressively move
# 500 more samples out of the "held out" pool into the RAG index. Each step,
# the evaluation set grows because the held-out samples join the evaluation.

# Shuffle the training set once so the chunks are representative.
rng = np.random.RandomState(42)
perm = rng.permutation(len(TRAIN_FULL_TEXTS))
TRAIN_FULL_TEXTS = [TRAIN_FULL_TEXTS[i] for i in perm]
TRAIN_FULL_LABELS = [TRAIN_FULL_LABELS[i] for i in perm]

# Embed the full training set once (reuse embedder; this is cached by the
# previous run if the same object was used — otherwise recompute).
print("Embedding full training set once (needed for all corpus sizes)...")
E_train_full = embedder.get_embeddings(TRAIN_FULL_TEXTS)
print(f"   E_train_full.shape = {E_train_full.shape}")

steps = list(range(500, len(TRAIN_FULL_TEXTS) + 1, 500))
if steps[-1] != len(TRAIN_FULL_TEXTS):
    steps.append(len(TRAIN_FULL_TEXTS))

scaling_rows = []
for step in steps:
    corpus_texts  = TRAIN_FULL_TEXTS[:step]
    corpus_labels = TRAIN_FULL_LABELS[:step]
    E_corpus      = E_train_full[:step]

    # Held-out training samples join the evaluation set.
    held_texts   = TRAIN_FULL_TEXTS[step:]
    held_labels  = [label_mapping[t] for t in TRAIN_FULL_LABELS[step:]]  # all known
    held_original = TRAIN_FULL_LABELS[step:]

    eval_texts   = TEST_BASE_TEXTS + held_texts
    eval_labels  = TEST_BASE_LABELS + held_labels
    eval_original = TEST_BASE_ORIGINAL + held_original

    print(f"\n--- RAG size = {step:5d}  eval set = {len(eval_texts):5d} "
          f"(test={len(TEST_BASE_TEXTS)}, held-out={len(held_texts)}) ---")

    m = evaluate_proposed(E_corpus, corpus_labels, eval_texts, eval_labels,
                           beta=0.15, label=f"Proposed@{step}")
    m["corpus_size"] = step
    m["eval_size"]   = len(eval_texts)
    scaling_rows.append(m)

df_scaling = pd.DataFrame(scaling_rows)
print("\nRAG corpus scaling summary:")
print(df_scaling.to_string(index=False))

# ---------------------------------------------------------------------------
# MODEL COMPARISON at fixed corpus sizes
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("MODEL COMPARISON at selected corpus sizes")
print("=" * 80)

comparison_corpus_sizes = [500, 1000, 2000, 3500, len(TRAIN_FULL_TEXTS)]
comparison_rows = []

for step in comparison_corpus_sizes:
    corpus_texts  = TRAIN_FULL_TEXTS[:step]
    corpus_labels = TRAIN_FULL_LABELS[:step]
    E_corpus      = E_train_full[:step]
    held_texts    = TRAIN_FULL_TEXTS[step:]
    held_labels   = [label_mapping[t] for t in TRAIN_FULL_LABELS[step:]]
    held_original = TRAIN_FULL_LABELS[step:]
    eval_texts    = TEST_BASE_TEXTS + held_texts
    eval_labels   = TEST_BASE_LABELS + held_labels

    print(f"\n=== Corpus size {step} ===")

    for name, fn in [
        ("Proposed (two-signal, pseudo-cal)", lambda: evaluate_proposed(E_corpus, corpus_labels, eval_texts, eval_labels, 0.15, f"Proposed@{step}")),
        ("Prototype only",                    lambda: evaluate_prototype_only(E_corpus, corpus_labels, eval_texts, eval_labels, 0.15, f"Proto@{step}")),
        ("kNN-OOD",                           lambda: evaluate_knn_ood(E_corpus, corpus_labels, eval_texts, eval_labels, 0.15, f"kNN@{step}")),
        ("Random Forest + conf",              lambda: evaluate_rf(corpus_texts, corpus_labels, eval_texts, eval_labels, 0.15, f"RF@{step}")),
    ]:
        try:
            m = fn()
            comparison_rows.append({
                "corpus_size": step, "model": name,
                "known_accuracy": m["known_accuracy"],
                "unknown_recall": m["unknown_recall"],
                "FAR": m["FAR"], "FRR": m["FRR"],
                "n_known": m["n_known"], "n_unknown": m["n_unknown"],
            })
            print(f"   {name:38s}  known_acc={m['known_accuracy']:.4f}  "
                  f"unknown_recall={m['unknown_recall']:.4f}  "
                  f"FAR={m['FAR']:.4f}  FRR={m['FRR']:.4f}")
        except Exception as e:
            print(f"   {name:38s}  FAILED: {e}")

df_comparison = pd.DataFrame(comparison_rows)
print("\nModel comparison summary:")
print(df_comparison.to_string(index=False))

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
out = {
    "rag_scaling": scaling_rows,
    "model_comparison": comparison_rows,
}
with open("benchmark_corpus_scaling.json", "w") as f:
    json.dump(out, f, indent=2, default=str)

zip_path = "benchmark_corpus_scaling.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("benchmark_corpus_scaling.json")
print(f"\n[saved] {zip_path}")

try:
    from google.colab import files as _f
    _f.download(zip_path)
    print("[download] started")
except Exception as e:
    print(f"[note] {e}")

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

RAG CORPUS SCALING — 500 to 4500 in steps of 500
Embedding full training set once (needed for all corpus sizes)...
   E_train_full.shape = (4500, 768)

--- RAG size =   500  eval set =  4800 (test=800, held-out=4000) ---



--- RAG size =  1000  eval set =  4300 (test=800, held-out=3500) ---



--- RAG size =  1500  eval set =  3800 (test=800, held-out=3000) ---



--- RAG size =  2000  eval set =  3300 (test=800, held-out=2500) ---



--- RAG size =  2500  eval set =  2800 (test=800, held-out=2000) ---


...

   Prototype only                          known_acc=0.7772  unknown_recall=1.0000  FAR=0.0000  FRR=0.2228


   kNN-OOD                                 known_acc=0.7011  unknown_recall=1.0000  FAR=0.0000  FRR=0.2989
   Random Forest + conf                    FAILED: name 'RandomForestClassifier' is not defined

Model comparison summary:
 corpus_size                             model  known_accuracy  unknown_recall  FAR      FRR  n_known  n_unknown
         500 Proposed (two-signal, pseudo-cal)        0.882030             1.0  0.0 0.117970     4552        248
         500                    Prototype only        0.911248             1.0  0.0 0.088752     4552        248
         500                           kNN-OOD        0.836995             1.0  0.0 0.163005     4552        248
        1000 Proposed (two-signal, pseudo-cal)        0.926703             1.0  0.0 0.073297     4052        248
        1000                    Prototype only        0.933613             1.0  0.0 0.066387     4052        248
        1000                           kNN-OOD        0.820829             1.0  0.0 0.179171

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[download] started

DONE


In [7]:
# =============================================================================
# CODASPY 2027 AUDIT + REPAIR — SINGLE CELL (v3, corrected embedder)
# =============================================================================

# -----------------------------------------------------------------------------
# INSTALL
# -----------------------------------------------------------------------------
!pip install -q faiss-cpu sentence-transformers transformers datasets \
    scikit-learn torch pandas numpy matplotlib seaborn tqdm scipy \
    huggingface_hub safetensors

# -----------------------------------------------------------------------------
# IMPORTS
# -----------------------------------------------------------------------------
import os, sys, json, time, random, gc, shutil, zipfile, pickle, warnings, math, re
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Callable, Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModel, ModernBertConfig, ModernBertModel,
    ModernBertForSequenceClassification, TrainingArguments, Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score, f1_score, roc_curve, auc,
    precision_recall_curve, confusion_matrix,
    matthews_corrcoef, balanced_accuracy_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import beta as _beta, chi2 as _chi2, ks_2samp, wasserstein_distance
from safetensors.torch import load_file
from huggingface_hub import snapshot_download
import faiss
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------
GLOBAL_SEEDS = [42, 1, 7, 13, 23]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

OUT_ROOT = "./audit_outputs"
for sub in ["protocol_A_leaky", "protocol_B_strict", "ablation", "seeds",
            "calibration", "fusion", "scaling", "adversarial", "distribution",
            "cross_family", "figures", "tables"]:
    os.makedirs(os.path.join(OUT_ROOT, sub), exist_ok=True)

def set_seed(s: int):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)

set_seed(GLOBAL_SEEDS[0])

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 10})

print("=" * 80)
print("CODASPY 2027 AUDIT + REPAIR (v3, corrected embedder)")
print("=" * 80)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Seeds: {GLOBAL_SEEDS}")
print("=" * 80)

# -----------------------------------------------------------------------------
# SECTION 1 — DATASETS WITH THREE-WAY LABEL SPACE
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 1 — DATASETS (KNOWN / UNKNOWN_ATTACK / BENIGN)")
print("=" * 80)

try:
    from google.colab import files as _colab_files
    _HAVE_COLAB = True
except Exception:
    _HAVE_COLAB = False

def ensure_file(name, required=True):
    if os.path.exists(name): return name
    alt = f"./datasets/{name}"
    if os.path.exists(alt):
        shutil.copy(alt, name); return name
    if not _HAVE_COLAB:
        if required: raise RuntimeError(f"Missing {name} (not in Colab).")
        return None
    print(f"Upload {name}")
    up = _colab_files.upload()
    if not up and required:
        raise RuntimeError(f"Required: {name}")
    for fn in up.keys():
        os.makedirs("./datasets", exist_ok=True)
        dest = f"./datasets/{fn}"
        if os.path.exists(fn): shutil.move(fn, dest)
        if not os.path.exists(name): shutil.copy(dest, name)
    return name

TRAINING_PATH = ensure_file("training_dataset.csv")
MASTER_PATH   = ensure_file("master_security_dataset.csv")
SAMPLE_PATH   = ensure_file("sample_security_dataset.csv")

df_train  = pd.read_csv(TRAINING_PATH)
df_master = pd.read_csv(MASTER_PATH)
df_sample = pd.read_csv(SAMPLE_PATH)

KNOWN_ATTACKS = [
    'DNS Fast-Flux', 'DoS', 'DoS + Brute-Force',
    'FTP Brute-Force / Data Exfiltration', 'HTTP C2',
    'ICMP Flood', 'IRC C2', 'P2P / UDP Scan', 'Spam',
]
LABEL_UNKNOWN = len(KNOWN_ATTACKS)
LABEL_BENIGN  = len(KNOWN_ATTACKS) + 1
label_mapping = {a: i for i, a in enumerate(KNOWN_ATTACKS)}
label_mapping['UNKNOWN_ATTACK'] = LABEL_UNKNOWN
label_mapping['BENIGN']         = LABEL_BENIGN
id_to_label = {v: k for k, v in label_mapping.items()}

BENIGN_TOKENS = {"benign", "normal", "background"}
def _classify_row(attack_type: str) -> str:
    s = str(attack_type).strip().lower()
    if any(t in s for t in BENIGN_TOKENS): return "BENIGN"
    if attack_type in KNOWN_ATTACKS:       return "KNOWN"
    return "UNKNOWN_ATTACK"

df_sample["_class"] = df_sample["Attack Type"].apply(_classify_row)
df_sample_unknown = df_sample[df_sample["_class"] == "UNKNOWN_ATTACK"].copy()
df_sample_benign  = df_sample[df_sample["_class"] == "BENIGN"].copy()

print(f"Training rows : {len(df_train)}")
print(f"Sample classes: {dict(df_sample['_class'].value_counts())}")
print(f"Unknown rows  : {len(df_sample_unknown)}")
print(f"Benign rows   : {len(df_sample_benign)}")

test_records = []
for attack in KNOWN_ATTACKS:
    sub = df_train[df_train['Attack Type'] == attack]
    n = min(50, len(sub))
    for _, row in sub.sample(n=n, random_state=42).iterrows():
        test_records.append((row['Log'], attack, label_mapping[attack], "known"))
for _, row in df_sample_unknown.sample(
        n=min(200, len(df_sample_unknown)), random_state=42).iterrows():
    test_records.append((row['Log'], row['Attack Type'], LABEL_UNKNOWN, "unknown"))
for _, row in df_sample_benign.sample(
        n=min(200, len(df_sample_benign)), random_state=42).iterrows():
    test_records.append((row['Log'], row['Attack Type'], LABEL_BENIGN, "benign"))

TEST_TEXTS    = [r[0] for r in test_records]
TEST_LABELS   = np.array([r[2] for r in test_records])
TEST_ORIGINAL = [r[1] for r in test_records]
TEST_ROLE     = np.array([r[3] for r in test_records])

print(f"\nFIXED TEST SET: {len(TEST_TEXTS)}")
for role in ["known", "unknown", "benign"]:
    print(f"  {role:8s}: {int((TEST_ROLE==role).sum())}")

with open(f"{OUT_ROOT}/fixed_test_set.json", "w") as f:
    json.dump({"texts": TEST_TEXTS, "labels": TEST_LABELS.tolist(),
               "original": TEST_ORIGINAL, "role": TEST_ROLE.tolist(),
               "known_attacks": KNOWN_ATTACKS,
               "label_mapping": label_mapping}, f, indent=2)

# -----------------------------------------------------------------------------
# SECTION 2 — ENCODERS
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 2 — ENCODERS")
print("=" * 80)

FINETUNED_REPO = "ccaug/modernbert-IDS-Unknown"
LOCAL_DIR = "./modernbert_ids_unknown"
os.makedirs(LOCAL_DIR, exist_ok=True)

def load_finetuned_encoder():
    try:
        if not os.path.exists(os.path.join(LOCAL_DIR, "model.safetensors")):
            snapshot_download(repo_id=FINETUNED_REPO, local_dir=LOCAL_DIR,
                              local_dir_use_symlinks=False, resume_download=True)
        tok = AutoTokenizer.from_pretrained(LOCAL_DIR)
        cfg = ModernBertConfig.from_pretrained("answerdotai/ModernBERT-base")
        model = ModernBertModel(cfg)
        sd = load_file(os.path.join(LOCAL_DIR, "model.safetensors"))
        mapped, skipped = {}, 0
        for k, v in sd.items():
            k2 = k
            for pre in ("bert.", "model.", "modernbert."):
                if k2.startswith(pre): k2 = k2[len(pre):]; break
            if any(k2.startswith(p) for p in
                   ["classifier", "loss_fct", "head.", "temperature"]):
                skipped += 1; continue
            mapped[k2] = v
        missing, unexpected = model.load_state_dict(mapped, strict=False)
        assert len(missing) == 0, f"missing keys: {missing}"
        model = model.to(DEVICE).eval()
        print(f"[finetuned] loaded (skipped {skipped} head weights)")
        return tok, model
    except Exception as e:
        print(f"[FALLBACK] fine-tuned encoder unavailable ({e}); using base.")
        tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
        model = AutoModel.from_pretrained("answerdotai/ModernBERT-base").to(DEVICE).eval()
        return tok, model

finetuned_tokenizer, finetuned_model = load_finetuned_encoder()
base_tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
base_model = AutoModel.from_pretrained("answerdotai/ModernBERT-base").to(DEVICE).eval()
print("[base] loaded")

# -----------------------------------------------------------------------------
# SECTION 3 — EMBEDDING / PROTOTYPE / RETRIEVER
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 3 — EMBEDDING, PROTOTYPES, MULTI-SIGNAL RETRIEVER")
print("=" * 80)


class EmbeddingOnlyModel:
    """
    Embedding wrapper. get_embeddings(texts) ALWAYS returns exactly
    len(texts) rows. Batches are accumulated into a list and vstacked;
    they are never allowed to leak out one at a time.
    """
    def __init__(self, model, tokenizer, device, max_length=512):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.max_length = max_length
        self.model.eval()
        self._cache: Dict[str, np.ndarray] = {}

    @torch.no_grad()
    def get_embeddings(self, texts, batch_size=32, use_cache=True):
        texts = list(texts)
        if len(texts) == 0:
            d = getattr(self.model.config, "hidden_size", 768)
            return np.zeros((0, d), dtype=np.float32)

        if use_cache:
            miss = [t for t in texts if t not in self._cache]
        else:
            miss = texts

        # Accumulate chunks for the misses (used only when use_cache=False).
        chunks = []
        for i in range(0, len(miss), batch_size):
            batch = miss[i:i + batch_size]
            inputs = self.tokenizer(
                batch, return_tensors="pt", truncation=True,
                max_length=self.max_length, padding=True,
            ).to(self.device)
            out = self.model(**inputs)
            last = out.last_hidden_state
            mask = inputs["attention_mask"].unsqueeze(-1)
            emb = (last * mask).sum(1) / mask.sum(1).clamp(min=1)
            emb = F.normalize(emb, dim=1)
            emb_np = emb.cpu().numpy()
            chunks.append(emb_np)
            # Always populate cache so subsequent calls are cheap.
            for t, e in zip(batch, emb_np):
                self._cache[t] = e

        # Serve in caller order. This is the only correct way to
        # guarantee len(result) == len(texts).
        return np.stack([self._cache[t] for t in texts])

    def clear_cache(self):
        self._cache.clear()


class PrototypeClassifier:
    def __init__(self, class_prototypes):
        self.class_names = sorted(class_prototypes.keys())
        M = np.stack([class_prototypes[c] for c in self.class_names])
        M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-8)
        self.prototype_matrix = M
    def _sims(self, q):
        q = q / (np.linalg.norm(q) + 1e-8)
        s = self.prototype_matrix @ q
        s = np.clip(s, -1.0, 1.0)
        return (s + 1.0) / 2.0
    def detect_unknown(self, q):
        s = self._sims(q); i = int(np.argmax(s))
        return self.class_names[i], float(s[i])
    def best_proto_similarity(self, q):
        return float(self._sims(q).max())


def _as_np_labels(y):
    return np.array([str(v) for v in y], dtype=object)


def build_prototypes(E, y):
    """E: (N, D). y: length-N sequence of labels. Hard-fails on mismatch."""
    E = np.asarray(E)
    y = _as_np_labels(y)
    if E.shape[0] != y.shape[0]:
        raise ValueError(
            f"build_prototypes: E has {E.shape[0]} rows but y has "
            f"{y.shape[0]} labels. An embedder returned a partial batch."
        )
    protos = {}
    for cls in sorted(set(y.tolist())):
        m = (y == cls)
        ce = E[m]
        if len(ce):
            p = ce.mean(0)
            protos[cls] = p / (np.linalg.norm(p) + 1e-8)
    return protos


def build_faiss_index(E):
    e = np.asarray(E, dtype="float32").copy()
    if e.ndim == 1: e = e.reshape(1, -1)
    faiss.normalize_L2(e)
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx


class MultiSignalRetriever:
    def __init__(self, index, doc_embeddings, doc_labels, pc, k=30):
        self.index = index
        self.doc_embeddings = np.asarray(doc_embeddings)
        self.doc_labels = _as_np_labels(doc_labels)
        if self.doc_embeddings.shape[0] != len(self.doc_labels):
            raise ValueError(
                f"Retriever: docs={self.doc_embeddings.shape[0]} "
                f"labels={len(self.doc_labels)}"
            )
        self.pc = pc
        self.k = min(k, len(self.doc_embeddings))
        self.doc_self_sim = np.array(
            [self.pc.best_proto_similarity(e) for e in self.doc_embeddings])
    def query(self, q):
        q = q / (np.linalg.norm(q) + 1e-8)
        sims, idxs = self.index.search(q.reshape(1, -1).astype('float32'), self.k)
        sims = (sims[0] + 1.0) / 2.0
        idxs = idxs[0]
        valid = [(s, i) for s, i in zip(sims, idxs)
                 if 0 <= i < len(self.doc_embeddings)]
        if not valid:
            return 0.0, 0.0, 1.0, 1.0, 1.0
        v_sims = np.array([v[0] for v in valid])
        v_idx  = [v[1] for v in valid]
        s_ret  = float(v_sims.mean())
        u_disp = float(v_sims.std())
        q_sim  = self.pc.best_proto_similarity(q)
        med    = float(np.median(self.doc_self_sim[v_idx]))
        local_outlier = q_sim / (med + 1e-8)
        neighbor_labels = self.doc_labels[v_idx]
        q_label = self.pc.detect_unknown(q)[0]
        disagree = float(np.mean(neighbor_labels != q_label))
        return q_sim, s_ret, u_disp, local_outlier, disagree


embedder_ft = EmbeddingOnlyModel(finetuned_model, finetuned_tokenizer, DEVICE)
embedder_base = EmbeddingOnlyModel(base_model, base_tokenizer, DEVICE)

# Sanity check before anything else: catch embedder regressions early.
_probe = embedder_ft.get_embeddings(TEST_TEXTS[:200], batch_size=32, use_cache=False)
assert _probe.shape[0] == 200, \
    f"[FATAL] embedder returned {_probe.shape[0]} rows for 200 texts."
print(f"[CHECK] embedder returns full arrays: {_probe.shape}")

# -----------------------------------------------------------------------------
# SECTION 4 — STRICT LOFO (PROTOCOL B) AND LEAKY LOFO (PROTOCOL A)
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 4 — LOFO PROTOCOLS")
print("=" * 80)

class SequenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = list(texts); self.labels = list(labels)
        self.tok = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True,
                       max_length=self.max_len, padding=False)
        enc["labels"] = int(self.labels[i]); return enc


def train_encoder_strict(train_texts, train_labels, seed=42, epochs=2,
                         batch_size=16, lr=2e-5, out_dir=None):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    uniq = sorted(set(train_labels)); l2i = {c: i for i, c in enumerate(uniq)}
    y_ids = [l2i[c] for c in train_labels]
    model = ModernBertForSequenceClassification.from_pretrained(
        "answerdotai/ModernBERT-base", num_labels=len(uniq)).to(DEVICE)
    ds = SequenceDataset(train_texts, y_ids, tok)
    collator = DataCollatorWithPadding(tokenizer=tok)

    steps_per_epoch = max(1, len(ds) // batch_size)
    total_steps = steps_per_epoch * epochs
    warmup_steps = max(1, int(0.06 * total_steps))

    args = TrainingArguments(
        output_dir=out_dir or "./strict_encoder_tmp",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
        seed=seed,
        fp16=torch.cuda.is_available(),
        disable_tqdm=True,
    )
    Trainer(model=model, args=args, train_dataset=ds,
            data_collator=collator).train()
    encoder = model.bert if hasattr(model, "bert") else model.model
    encoder = encoder.to(DEVICE).eval()
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        encoder.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return tok, encoder, l2i


@dataclass
class LOFOResult:
    held_family: str
    n_train: int
    n_held: int
    pseudo_known: np.ndarray
    pseudo_unknown: np.ndarray
    held_test: np.ndarray
    tau_p: float = 0.0
    tau_u: float = 1.0
    protocol: str = ""
    pseudo_frr: float = 1.0
    pseudo_rec: float = 0.0


def _nested_pseudo(E_tr, y_tr, k):
    E_tr = np.asarray(E_tr)
    y_tr = _as_np_labels(y_tr)
    if E_tr.shape[0] != y_tr.shape[0]:
        raise ValueError(
            f"_nested_pseudo: E={E_tr.shape[0]} y={y_tr.shape[0]}"
        )
    pk, pu = [], []
    for inner in sorted(set(y_tr.tolist())):
        mask = (y_tr == inner)
        if int(mask.sum()) < 2: continue
        E_in_tr = E_tr[~mask]; y_in_tr = y_tr[~mask]
        E_in_te = E_tr[mask]
        if len(E_in_tr) < 5: continue
        protos_in = build_prototypes(E_in_tr, y_in_tr)
        pc_in = PrototypeClassifier(protos_in)
        idx_in = build_faiss_index(E_in_tr)
        retr_in = MultiSignalRetriever(idx_in, E_in_tr, y_in_tr, pc_in, k=k)
        for e in E_in_te[:80]:
            pu.append(retr_in.query(e))
        for e in E_in_tr[::6]:
            pk.append(retr_in.query(e))
    if not pk: pk = [np.zeros(5)]
    if not pu: pu = [np.zeros(5)]
    return np.array(pk), np.array(pu)


def _calibrate_2sig(pk, pu, beta=0.15):
    if len(pk) == 0 or len(pu) == 0:
        return 0.5, 0.5, 1.0, 0.0
    s_p_K, u_d_K = pk[:, 0], pk[:, 2]
    s_p_U, u_d_U = pu[:, 0], pu[:, 2]
    gp = np.linspace(np.percentile(s_p_K, 50), np.percentile(s_p_K, 99), 20)
    gu = np.linspace(np.percentile(u_d_K, 1),  np.percentile(u_d_K, 99), 20)
    best = None
    for tp in gp:
        for tu in gu:
            frr = ((s_p_K < tp) | (u_d_K > tu)).mean()
            if frr > beta: continue
            rec = ((s_p_U < tp) | (u_d_U > tu)).mean()
            key = (round(rec, 4), -round(frr, 4), round(tp, 4))
            if best is None or key > best[0]:
                best = (key, tp, tu, frr, rec)
    if best is None:
        return float(np.percentile(s_p_K, 5)), float(np.percentile(u_d_K, 95)), 1.0, 0.0
    _, tp, tu, frr, rec = best
    return float(tp), float(tu), float(frr), float(rec)


def strict_lofo_run(df_train, known_attacks, k=30, beta=0.15, seed=42,
                    retrain=False, cache_dir=f"{OUT_ROOT}/protocol_B_strict"):
    results = []
    for fam in known_attacks:
        print(f"  [B] holding out {fam}")
        tr = df_train[df_train['Attack Type'] != fam].reset_index(drop=True)
        te = df_train[df_train['Attack Type'] == fam].reset_index(drop=True)
        if len(te) > 200:
            te = te.sample(n=200, random_state=seed).reset_index(drop=True)

        enc_dir = os.path.join(cache_dir,
                               f"enc_{fam.replace(' ','_').replace('/','_')}")
        # Load or train encoder
        try:
            if retrain or not os.path.exists(os.path.join(enc_dir, "model.safetensors")):
                tok, enc, _ = train_encoder_strict(
                    tr['Log'].tolist(), tr['Attack Type'].tolist(),
                    seed=seed, out_dir=enc_dir)
            else:
                tok = AutoTokenizer.from_pretrained(enc_dir)
                enc = ModernBertModel.from_pretrained(enc_dir).to(DEVICE).eval()
            emb = EmbeddingOnlyModel(enc, tok, DEVICE)
        except Exception as e:
            print(f"    [FALLBACK] strict encoder failed ({e}); using base.")
            emb = EmbeddingOnlyModel(base_model, base_tokenizer, DEVICE)

        E_tr = emb.get_embeddings(tr['Log'].tolist(), use_cache=False)
        y_tr = tr['Attack Type'].tolist()
        E_te = emb.get_embeddings(te['Log'].tolist(), use_cache=False)

        # Hard guard. Never truncate.
        if E_tr.shape[0] != len(y_tr):
            raise RuntimeError(
                f"strict_lofo_run[{fam}]: embedder returned "
                f"{E_tr.shape[0]} rows for {len(y_tr)} texts."
            )
        if E_te.shape[0] != len(te):
            raise RuntimeError(
                f"strict_lofo_run[{fam}]: held-out embedder returned "
                f"{E_te.shape[0]} rows for {len(te)} texts."
            )

        protos = build_prototypes(E_tr, y_tr)
        pc = PrototypeClassifier(protos)
        idx = build_faiss_index(E_tr)
        retr = MultiSignalRetriever(idx, E_tr, y_tr, pc, k=k)

        pk, pu = _nested_pseudo(E_tr, y_tr, k)
        held_scores = np.array([retr.query(e) for e in E_te])
        tp, tu, frr, rec = _calibrate_2sig(pk, pu, beta)

        results.append(LOFOResult(
            held_family=fam, n_train=len(tr), n_held=len(te),
            pseudo_known=pk, pseudo_unknown=pu,
            held_test=held_scores, tau_p=tp, tau_u=tu,
            protocol="B_strict", pseudo_frr=frr, pseudo_rec=rec))
        print(f"     tau_p={tp:.4f} tau_u={tu:.4f} "
              f"pseudo_FRR={frr:.4f} pseudo_rec={rec:.4f} "
              f"| pk={len(pk)} pu={len(pu)}")

        del emb, retr, enc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return results


def protocol_A_lofo(df_train, known_attacks, k=30, beta=0.15, embedder=None):
    results = []
    for fam in known_attacks:
        print(f"  [A] holding out {fam}")
        tr = df_train[df_train['Attack Type'] != fam].reset_index(drop=True)
        te = df_train[df_train['Attack Type'] == fam].reset_index(drop=True)
        if len(te) > 200:
            te = te.sample(n=200, random_state=42).reset_index(drop=True)

        E_tr = embedder.get_embeddings(tr['Log'].tolist())
        y_tr = tr['Attack Type'].tolist()
        E_te = embedder.get_embeddings(te['Log'].tolist())

        if E_tr.shape[0] != len(y_tr):
            raise RuntimeError(
                f"protocol_A_lofo[{fam}]: embedder returned "
                f"{E_tr.shape[0]} rows for {len(y_tr)} texts."
            )

        protos = build_prototypes(E_tr, y_tr)
        pc = PrototypeClassifier(protos)
        idx = build_faiss_index(E_tr)
        retr = MultiSignalRetriever(idx, E_tr, y_tr, pc, k=k)
        pk, pu = _nested_pseudo(E_tr, y_tr, k)
        held_scores = np.array([retr.query(e) for e in E_te])
        tp, tu, frr, rec = _calibrate_2sig(pk, pu, beta)
        results.append(LOFOResult(
            held_family=fam, n_train=len(tr), n_held=len(te),
            pseudo_known=pk, pseudo_unknown=pu,
            held_test=held_scores, tau_p=tp, tau_u=tu,
            protocol="A_leaky", pseudo_frr=frr, pseudo_rec=rec))
        print(f"     tau_p={tp:.4f} tau_u={tu:.4f} "
              f"pseudo_FRR={frr:.4f} pseudo_rec={rec:.4f} "
              f"| pk={len(pk)} pu={len(pu)}")
    return results


print("\nRunning Protocol A (encoder-leaky)...")
res_A = protocol_A_lofo(df_train, KNOWN_ATTACKS, k=30, beta=0.15,
                        embedder=embedder_ft)
embedder_ft.clear_cache()

print("\nRunning Protocol B (strict LOFO, retrain=False uses cached encoders)...")
res_B = strict_lofo_run(df_train, KNOWN_ATTACKS, k=30, beta=0.15,
                        seed=42, retrain=False)


def summarize_lofo(results, name):
    rows = []
    for r in results:
        if len(r.held_test) == 0: continue
        rej = ((r.held_test[:, 0] < r.tau_p) |
               (r.held_test[:, 2] > r.tau_u)).astype(float)
        rows.append({"family": r.held_family, "n_held": r.n_held,
                     "rejection_rate": float(rej.mean()),
                     "tau_p": r.tau_p, "tau_u": r.tau_u,
                     "pseudo_frr": r.pseudo_frr,
                     "pseudo_rec": r.pseudo_rec})
    df = pd.DataFrame(rows)
    print(f"\n{name}")
    if len(df):
        print(df.to_string(index=False))
        print(f"  mean={df['rejection_rate'].mean():.4f}  "
              f"min={df['rejection_rate'].min():.4f}  "
              f"max={df['rejection_rate'].max():.4f}")
    return df


df_A = summarize_lofo(res_A, "Protocol A — held-out rejection (leaky encoder)")
df_B = summarize_lofo(res_B, "Protocol B — held-out rejection (strict LOFO)")

leak_gap = (df_A['rejection_rate'].mean() - df_B['rejection_rate'].mean()) \
           if len(df_A) and len(df_B) else float('nan')
print(f"\n>>> LEAKAGE GAP (A - B) = {leak_gap:+.4f}")
print("    A large positive gap = the fine-tuned encoder had seen the held family.")

if len(df_A) and len(df_B):
    fig, ax = plt.subplots(figsize=(10, 5))
    fams = sorted(set(df_A['family']).union(set(df_B['family'])))
    a = df_A.set_index('family').reindex(fams)['rejection_rate'].values
    b = df_B.set_index('family').reindex(fams)['rejection_rate'].values
    x = np.arange(len(fams)); w = 0.38
    ax.bar(x - w/2, np.nan_to_num(a), w, label='Protocol A (leaky)')
    ax.bar(x + w/2, np.nan_to_num(b), w, label='Protocol B (strict)')
    ax.set_xticks(x); ax.set_xticklabels(fams, rotation=30, ha='right')
    ax.set_ylabel('Held-out family rejection rate')
    ax.set_title('Encoder leakage: leaky vs strict LOFO')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT_ROOT}/figures/fig01_leakage_gap.png"); plt.close()

df_A.to_csv(f"{OUT_ROOT}/protocol_A_leaky/per_family.csv", index=False)
df_B.to_csv(f"{OUT_ROOT}/protocol_B_strict/per_family.csv", index=False)

# -----------------------------------------------------------------------------
# SECTION 5 — FUSION ABLATION
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 5 — FUSION STRATEGY ABLATION")
print("=" * 80)

def _stack_pseudo(res):
    K_list = [r.pseudo_known for r in res if len(r.pseudo_known)]
    U_list = [r.pseudo_unknown for r in res if len(r.pseudo_unknown)]
    K = np.vstack(K_list) if K_list else np.zeros((1, 5))
    U = np.vstack(U_list) if U_list else np.zeros((1, 5))
    return K, U

K, U = _stack_pseudo(res_B if len(res_B) else res_A)
print(f"Stacked pseudo-known: {len(K)}   pseudo-unknown: {len(U)}")
assert len(K) > 100 and len(U) > 100, \
    "Pseudo distributions are too small; embedder likely still broken."


def _or_rule(S, tau):
    return ((S[:, 0] < tau["tp"]) | (S[:, 2] > tau["tu"])).astype(float)
def _and_rule(S, tau):
    return ((S[:, 0] < tau["tp"]) & (S[:, 2] > tau["tu"])).astype(float)
def _weighted_rule(S, tau):
    w = tau["w"]; z = w[0] * (1 - S[:, 0]) + w[1] * S[:, 2]
    return (z > tau["z"]).astype(float)
def _logistic_rule(S, tau):
    return (tau["clf"].predict_proba(S[:, [0, 2, 3, 4]])[:, 1] > 0.5).astype(float)


def _fit_calibrated_fusion(K, U, beta=0.15):
    s_p_K, u_d_K = K[:, 0], K[:, 2]
    s_p_U, u_d_U = U[:, 0], U[:, 2]
    out = {}
    gp = np.linspace(np.percentile(s_p_K, 50), np.percentile(s_p_K, 99), 20)
    gu = np.linspace(np.percentile(u_d_K, 1),  np.percentile(u_d_K, 99), 20)
    # OR
    best = None
    for tp in gp:
        for tu in gu:
            frr = ((s_p_K < tp) | (u_d_K > tu)).mean()
            if frr > beta: continue
            rec = ((s_p_U < tp) | (u_d_U > tu)).mean()
            key = (round(rec, 4), -round(frr, 4))
            if best is None or key > best[0]: best = (key, tp, tu)
    if best:
        out["OR"] = {"tp": float(best[1]), "tu": float(best[2]),
                     "pseudo_frr": float(((s_p_K < best[1]) | (u_d_K > best[2])).mean()),
                     "pseudo_rec": float(((s_p_U < best[1]) | (u_d_U > best[2])).mean())}
    # AND
    best = None
    for tp in gp:
        for tu in gu:
            frr = ((s_p_K < tp) & (u_d_K > tu)).mean()
            if frr > beta: continue
            rec = ((s_p_U < tp) & (u_d_U > tu)).mean()
            key = (round(rec, 4), -round(frr, 4))
            if best is None or key > best[0]: best = (key, tp, tu)
    if best:
        out["AND"] = {"tp": float(best[1]), "tu": float(best[2]),
                      "pseudo_frr": float(((s_p_K < best[1]) & (u_d_K > best[2])).mean()),
                      "pseudo_rec": float(((s_p_U < best[1]) & (u_d_U > best[2])).mean())}
    # WEIGHTED
    best = None
    for w1 in np.linspace(0, 1, 11):
        w = (w1, 1 - w1)
        zK = w[0] * (1 - s_p_K) + w[1] * u_d_K
        zU = w[0] * (1 - s_p_U) + w[1] * u_d_U
        for zt in np.linspace(np.percentile(zK, 50), np.percentile(zK, 99), 20):
            frr = (zK > zt).mean()
            if frr > beta: continue
            rec = (zU > zt).mean()
            key = (round(rec, 4), -round(frr, 4))
            if best is None or key > best[0]: best = (key, w, zt)
    if best:
        out["WEIGHTED"] = {"w": best[1], "z": float(best[2]),
                           "pseudo_frr": float((best[1][0] * (1 - s_p_K) + best[1][1] * u_d_K > best[2]).mean()),
                           "pseudo_rec": float((best[1][0] * (1 - s_p_U) + best[1][1] * u_d_U > best[2]).mean())}
    # LOGISTIC
    Xp = np.vstack([K[:, [0, 2, 3, 4]], U[:, [0, 2, 3, 4]]])
    yp = np.array([0] * len(K) + [1] * len(U))
    clf = LogisticRegression(max_iter=2000).fit(Xp, yp)
    pK = clf.predict_proba(K[:, [0, 2, 3, 4]])[:, 1]
    pU = clf.predict_proba(U[:, [0, 2, 3, 4]])[:, 1]
    th = None
    for t in np.linspace(0.01, 0.99, 99):
        if (pK > t).mean() <= beta: th = t; break
    if th is None: th = 0.5
    out["LOGISTIC"] = {"th": float(th),
                       "pseudo_frr": float((pK > th).mean()),
                       "pseudo_rec": float((pU > th).mean()),
                       "clf": clf}
    return out


fusion_params = _fit_calibrated_fusion(K, U, beta=0.15)
print("\nFusion strategies calibrated on identical pseudo data:")
for name, p in fusion_params.items():
    print(f"  {name:10s} pseudo_FRR={p['pseudo_frr']:.4f}  "
          f"pseudo_rec={p['pseudo_rec']:.4f}")


def _apply_fusion(S, name, params):
    if name == "OR":       return _or_rule(S, {"tp": params["tp"], "tu": params["tu"]})
    if name == "AND":      return _and_rule(S, {"tp": params["tp"], "tu": params["tu"]})
    if name == "WEIGHTED": return _weighted_rule(S, {"w": params["w"], "z": params["z"]})
    if name == "LOGISTIC": return _logistic_rule(S, {"clf": params["clf"]})
    raise ValueError(name)


E_test_ft = embedder_ft.get_embeddings(TEST_TEXTS)
E_train_ft = embedder_ft.get_embeddings(df_train['Log'].tolist())
y_train_ft = df_train['Attack Type'].tolist()
protos_ft = build_prototypes(E_train_ft, y_train_ft)
pc_ft = PrototypeClassifier(protos_ft)
idx_ft = build_faiss_index(E_train_ft)
retr_ft = MultiSignalRetriever(idx_ft, E_train_ft, y_train_ft, pc_ft, k=30)
TEST_SCORES_FT = np.array([retr_ft.query(e) for e in E_test_ft])
print(f"\nFixed test set scored: {TEST_SCORES_FT.shape}")

km = (TEST_LABELS != LABEL_UNKNOWN)
um = (TEST_LABELS == LABEL_UNKNOWN)
bm = (TEST_LABELS == LABEL_BENIGN)

fusion_rows = []
for name, p in fusion_params.items():
    flags = _apply_fusion(TEST_SCORES_FT, name, p)
    unknown_recall = float(flags[um].mean()) if um.any() else float("nan")
    known_acc = float((1 - flags[km]).mean()) if km.any() else float("nan")
    far = 1.0 - unknown_recall
    frr = float(flags[km].mean()) if km.any() else float("nan")
    benign_rej = float(flags[bm].mean()) if bm.any() else float("nan")
    fusion_rows.append({"fusion": name, "known_acc": known_acc,
                        "unknown_recall": unknown_recall, "FAR": far,
                        "FRR": frr, "benign_rejection": benign_rej})

df_fusion = pd.DataFrame(fusion_rows)
print("\nFusion strategies on FIXED test set:")
print(df_fusion.to_string(index=False))
df_fusion.to_csv(f"{OUT_ROOT}/fusion/fusion_ablation.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(df_fusion)); w = 0.25
ax.bar(x - w, df_fusion['unknown_recall'], w, label='Unknown recall')
ax.bar(x, df_fusion['FRR'], w, label='FRR (known)')
ax.bar(x + w, df_fusion['benign_rejection'], w, label='Benign rejection')
ax.set_xticks(x); ax.set_xticklabels(df_fusion['fusion'])
ax.set_title('Fusion strategy comparison (fixed test set)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_ROOT}/figures/fig02_fusion.png"); plt.close()

# -----------------------------------------------------------------------------
# SECTION 6 — SIGNAL ABLATION MATRIX
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 6 — SIGNAL ABLATION MATRIX")
print("=" * 80)

def _eval_signal_subset(Kp, Up, cols, beta=0.15):
    Ks = Kp[:, cols]; Us = Up[:, cols]
    mins = Ks.min(0); maxs = Ks.max(0)
    if np.any(maxs - mins <= 1e-12):
        return None
    def _norm(X): return (X - mins) / (maxs - mins + 1e-9)
    Kn = _norm(Ks); Un = _norm(Us)
    best = None
    for t in np.linspace(0.05, 0.95, 30):
        frr = (Kn < t).any(1).mean()
        if frr > beta: continue
        rec = (Un < t).any(1).mean()
        key = (round(rec, 4), -round(frr, 4))
        if best is None or key > best[0]: best = (key, t, mins, maxs)
    if best is None: return None
    return best[1], best[2], best[3]

def _apply_signal_subset(S, cols, t, mins, maxs):
    Xs = S[:, cols]
    Xn = (Xs - mins) / (maxs - mins + 1e-9)
    return (Xn < t).any(1).astype(float)

signal_names = ["prototype", "retrieval", "dispersion", "local_outlier", "disagreement"]
signal_subsets = [
    [0], [0, 1], [0, 2], [0, 3], [0, 4],
    [0, 1, 2], [0, 2, 3], [0, 2, 4], [0, 3, 4],
    [0, 1, 2, 3, 4],
]
ablation_rows = []
for cols in signal_subsets:
    res = _eval_signal_subset(K, U, cols, beta=0.15)
    if res is None: continue
    t, mins, maxs = res
    flags = _apply_signal_subset(TEST_SCORES_FT, cols, t, mins, maxs)
    unknown_recall = float(flags[um].mean())
    known_acc = float((1 - flags[km]).mean())
    frr = float(flags[km].mean())
    name = "+".join(signal_names[c] for c in cols)
    ablation_rows.append({"signals": name, "known_acc": known_acc,
                          "unknown_recall": unknown_recall,
                          "FAR": 1 - unknown_recall, "FRR": frr})

if ablation_rows:
    df_abl = pd.DataFrame(ablation_rows)
    print(df_abl.to_string(index=False))
    df_abl.to_csv(f"{OUT_ROOT}/ablation/signal_ablation.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(df_abl))
    ax.plot(x, df_abl['unknown_recall'], 'o-', label='Unknown recall')
    ax.plot(x, df_abl['known_acc'],    's-', label='Known accuracy')
    ax.plot(x, df_abl['FRR'],          '^-', label='FRR')
    ax.set_xticks(x); ax.set_xticklabels(df_abl['signals'], rotation=45, ha='right')
    ax.set_title('Signal ablation matrix (calibrated at beta=0.15)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT_ROOT}/figures/fig03_ablation.png"); plt.close()
else:
    df_abl = pd.DataFrame(columns=["signals", "known_acc", "unknown_recall", "FAR", "FRR"])
    print("[WARN] no ablation rows produced.")

# -----------------------------------------------------------------------------
# SECTION 7 — CALIBRATION ABLATION WITH ORACLE
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 7 — CALIBRATION ABLATION (incl. ORACLE)")
print("=" * 80)

cal_rows = []

# (a) Arbitrary thresholds
for tp_a, tu_a in [(0.3, 0.5), (0.5, 0.5), (0.7, 0.5)]:
    flags = ((TEST_SCORES_FT[:, 0] < tp_a) | (TEST_SCORES_FT[:, 2] > tu_a)).astype(float)
    cal_rows.append({"calibration": f"arbitrary tp={tp_a} tu={tu_a}",
                     "unknown_recall": float(flags[um].mean()),
                     "FAR": 1 - float(flags[um].mean()),
                     "FRR": float(flags[km].mean()),
                     "used_real_unknown": False})

# (b) Quantile on known only
s_p_train = np.array([pc_ft.best_proto_similarity(e) for e in E_train_ft])
tp_q = float(np.percentile(s_p_train, 5))
flags = (TEST_SCORES_FT[:, 0] < tp_q).astype(float)
cal_rows.append({"calibration": "quantile on known",
                 "unknown_recall": float(flags[um].mean()),
                 "FAR": 1 - float(flags[um].mean()),
                 "FRR": float(flags[km].mean()),
                 "used_real_unknown": False})

# (c) Synthetic Gaussian OOD
rng_s = np.random.RandomState(0)
synth = rng_s.randn(1000, E_train_ft.shape[1]).astype("float32")
synth = synth / np.linalg.norm(synth, axis=1, keepdims=True)
proto_synth = build_prototypes(np.vstack([E_train_ft, synth]),
                                y_train_ft + ["__SYNTH__"] * len(synth))
pc_synth = PrototypeClassifier(proto_synth)
idx_synth = build_faiss_index(np.vstack([E_train_ft, synth]))
retr_synth = MultiSignalRetriever(idx_synth, np.vstack([E_train_ft, synth]),
                                  y_train_ft + ["__SYNTH__"] * len(synth), pc_synth, k=30)
synth_scores = np.array([retr_synth.query(e) for e in synth[:500]])
tp_syn = float(np.percentile(synth_scores[:, 0], 95))
tu_syn = float(np.percentile(synth_scores[:, 2], 5))
flags = ((TEST_SCORES_FT[:, 0] < tp_syn) | (TEST_SCORES_FT[:, 2] > tu_syn)).astype(float)
cal_rows.append({"calibration": "synthetic Gaussian OOD",
                 "unknown_recall": float(flags[um].mean()),
                 "FAR": 1 - float(flags[um].mean()),
                 "FRR": float(flags[km].mean()),
                 "used_real_unknown": False})

# (d) Pseudo-unknown (ours)
tp_ours, tu_ours, _, _ = _calibrate_2sig(K, U, beta=0.15)
flags = ((TEST_SCORES_FT[:, 0] < tp_ours) | (TEST_SCORES_FT[:, 2] > tu_ours)).astype(float)
cal_rows.append({"calibration": "pseudo-unknown (ours)",
                 "unknown_recall": float(flags[um].mean()),
                 "FAR": 1 - float(flags[um].mean()),
                 "FRR": float(flags[km].mean()),
                 "used_real_unknown": False})

# (e) ORACLE
s_p_u_real = TEST_SCORES_FT[um, 0]; u_d_u_real = TEST_SCORES_FT[um, 2]
tp_oracle = float(np.percentile(s_p_u_real, 99))
tu_oracle = float(np.percentile(u_d_u_real, 1))
flags = ((TEST_SCORES_FT[:, 0] < tp_oracle) | (TEST_SCORES_FT[:, 2] > tu_oracle)).astype(float)
cal_rows.append({"calibration": "ORACLE (real unknown)",
                 "unknown_recall": float(flags[um].mean()),
                 "FAR": 1 - float(flags[um].mean()),
                 "FRR": float(flags[km].mean()),
                 "used_real_unknown": True})

df_cal = pd.DataFrame(cal_rows)
print(df_cal.to_string(index=False))
df_cal.to_csv(f"{OUT_ROOT}/calibration/calibration_ablation.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_cal)); w = 0.3
ax.bar(x - w, df_cal['unknown_recall'], w, label='Unknown recall')
ax.bar(x,      df_cal['FAR'], w, label='FAR')
ax.bar(x + w,  df_cal['FRR'], w, label='FRR')
ax.set_xticks(x); ax.set_xticklabels(df_cal['calibration'], rotation=30, ha='right')
ax.set_title('Calibration ablation (ORACLE uses real unknown)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_ROOT}/figures/fig04_calibration.png"); plt.close()

# -----------------------------------------------------------------------------
# SECTION 8 — MULTI-SEED STABILITY + FAMILY-LEVEL BOOTSTRAP
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 8 — MULTI-SEED STABILITY + FAMILY-LEVEL BOOTSTRAP")
print("=" * 80)

flags_primary = ((TEST_SCORES_FT[:, 0] < tp_ours) |
                 (TEST_SCORES_FT[:, 2] > tu_ours)).astype(float)

seed_rows = []
for seed in GLOBAL_SEEDS:
    set_seed(seed)
    rng_seed = np.random.RandomState(seed)
    nK, nU = len(K), len(U)
    selK = rng_seed.choice(nK, size=max(1, int(0.8 * nK)), replace=False)
    selU = rng_seed.choice(nU, size=max(1, int(0.8 * nU)), replace=False)
    tp_s, tu_s, _, _ = _calibrate_2sig(K[selK], U[selU], beta=0.15)
    flags = ((TEST_SCORES_FT[:, 0] < tp_s) |
             (TEST_SCORES_FT[:, 2] > tu_s)).astype(float)
    seed_rows.append({
        "seed": seed,
        "known_acc": float((1 - flags[km]).mean()),
        "unknown_recall": float(flags[um].mean()),
        "FAR": 1 - float(flags[um].mean()),
        "FRR": float(flags[km].mean()),
        "benign_rejection": float(flags[bm].mean()),
        "tau_p": float(tp_s), "tau_u": float(tu_s),
    })

df_seeds = pd.DataFrame(seed_rows)
print(df_seeds.to_string(index=False))
print("\nMean +/- std across seeds:")
for c in ["known_acc", "unknown_recall", "FAR", "FRR", "benign_rejection"]:
    print(f"  {c:18s} = {df_seeds[c].mean():.4f} +/- {df_seeds[c].std():.4f}")
df_seeds.to_csv(f"{OUT_ROOT}/seeds/seed_stability.csv", index=False)


def family_bootstrap(flags, labels, original, n_boot=1000, seed=0):
    rng_b = np.random.RandomState(seed)
    orig = np.array(original)
    families = sorted(set(original))
    unk_fams = [f for f in families
                if np.any((orig == f) & (labels == LABEL_UNKNOWN))]
    if not unk_fams:
        return np.array([0.0])
    recs = []
    for _ in range(n_boot):
        sampled = rng_b.choice(unk_fams, size=len(unk_fams), replace=True)
        idx = []
        for f in sampled:
            idx.extend(np.where(orig == f)[0])
        if not idx: continue
        f_ = flags[np.array(idx)]; l_ = labels[np.array(idx)]
        if (l_ == LABEL_UNKNOWN).any():
            recs.append(float(f_[l_ == LABEL_UNKNOWN].mean()))
    return np.array(recs) if recs else np.array([0.0])


boot_recs = family_bootstrap(flags_primary, TEST_LABELS, TEST_ORIGINAL)
print(f"\nFamily-level bootstrap ({len(boot_recs)} resamples):")
print(f"  unknown recall  mean = {boot_recs.mean():.4f}  "
      f"95% CI = [{np.percentile(boot_recs, 2.5):.4f}, "
      f"{np.percentile(boot_recs, 97.5):.4f}]")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(df_seeds['seed'].astype(str), df_seeds['unknown_recall'])
axes[0].set_title('Unknown recall across seeds'); axes[0].set_ylim(0, 1.05)
axes[0].axhline(1.0, color='r', ls='--', alpha=0.4)
axes[1].hist(boot_recs, bins=30, edgecolor='k')
axes[1].axvline(1.0, color='r', ls='--', alpha=0.6)
axes[1].set_title('Family-level bootstrap of unknown recall')
plt.tight_layout(); plt.savefig(f"{OUT_ROOT}/figures/fig05_seeds.png"); plt.close()

# -----------------------------------------------------------------------------
# SECTION 9 — FIXED-TEST CORPUS SCALING
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 9 — CORPUS SCALING ON FIXED TEST SET")
print("=" * 80)

rng_p = np.random.RandomState(7)
perm = rng_p.permutation(len(df_train))
train_texts_all = df_train['Log'].tolist()
train_labels_all = df_train['Attack Type'].tolist()

scaling_rows = []
for size in [500, 1000, 2000, 3500, 4500]:
    size = min(size, len(train_texts_all))
    sub_idx = perm[:size]
    corpus_texts = [train_texts_all[i] for i in sub_idx]
    corpus_labels = [train_labels_all[i] for i in sub_idx]
    E_corpus = embedder_ft.get_embeddings(corpus_texts)
    protos = build_prototypes(E_corpus, corpus_labels)
    pc = PrototypeClassifier(protos)
    idx = build_faiss_index(E_corpus)
    retr = MultiSignalRetriever(idx, E_corpus, corpus_labels, pc, k=30)
    S = np.array([retr.query(e) for e in E_test_ft])
    pk, pu = _nested_pseudo(E_corpus, corpus_labels, 30)
    tp, tu, _, _ = _calibrate_2sig(pk, pu, beta=0.15)
    flags = ((S[:, 0] < tp) | (S[:, 2] > tu)).astype(float)
    scaling_rows.append({
        "corpus_size": size,
        "eval_size": len(TEST_TEXTS),
        "known_acc": float((1 - flags[km]).mean()),
        "unknown_recall": float(flags[um].mean()),
        "FAR": 1 - float(flags[um].mean()),
        "FRR": float(flags[km].mean()),
    })
    print(f"  size={size:5d}  known={scaling_rows[-1]['known_acc']:.4f}  "
          f"unk_rec={scaling_rows[-1]['unknown_recall']:.4f}  "
          f"FRR={scaling_rows[-1]['FRR']:.4f}")

df_scaling = pd.DataFrame(scaling_rows)
df_scaling.to_csv(f"{OUT_ROOT}/scaling/fixed_test_scaling.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_scaling['corpus_size'], df_scaling['known_acc'], 'o-', label='Known acc')
ax.plot(df_scaling['corpus_size'], df_scaling['unknown_recall'], 's-', label='Unknown recall')
ax.plot(df_scaling['corpus_size'], df_scaling['FRR'], '^-', label='FRR')
ax.set_xlabel('Retrieval corpus size'); ax.set_ylabel('Metric value')
ax.set_title('Corpus scaling on a FIXED test set')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_ROOT}/figures/fig06_scaling.png"); plt.close()

# -----------------------------------------------------------------------------
# SECTION 10 — DISTRIBUTION SIMILARITY
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 10 — PSEUDO vs REAL DISTRIBUTION SIMILARITY")
print("=" * 80)

real_unknown = TEST_SCORES_FT[um]
real_known = TEST_SCORES_FT[km]
pseudo_known = K
pseudo_unknown = U

signal_names_5 = ["s_proto", "s_ret", "u_disp", "local_outlier", "disagree"]
dist_rows = []
for i, sname in enumerate(signal_names_5):
    ks_k, p_k = ks_2samp(pseudo_known[:, i], real_known[:, i])
    w_k = wasserstein_distance(pseudo_known[:, i], real_known[:, i])
    ks_u, p_u = ks_2samp(pseudo_unknown[:, i], real_unknown[:, i])
    w_u = wasserstein_distance(pseudo_unknown[:, i], real_unknown[:, i])
    dist_rows.append({"signal": sname,
                      "KS_known": ks_k, "p_known": p_k, "W_known": w_k,
                      "KS_unknown": ks_u, "p_unknown": p_u, "W_unknown": w_u})
df_dist = pd.DataFrame(dist_rows)
print(df_dist.to_string(index=False))
df_dist.to_csv(f"{OUT_ROOT}/distribution/pseudo_vs_real.csv", index=False)

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, sname in enumerate(signal_names_5):
    axes[0, i].hist(pseudo_known[:, i], bins=30, alpha=0.5, label='pseudo-K', density=True)
    axes[0, i].hist(real_known[:, i], bins=30, alpha=0.5, label='real-K', density=True)
    axes[0, i].set_title(f"{sname}\nknown"); axes[0, i].legend(fontsize=7)
    axes[1, i].hist(pseudo_unknown[:, i], bins=30, alpha=0.5, label='pseudo-U', density=True)
    axes[1, i].hist(real_unknown[:, i], bins=30, alpha=0.5, label='real-U', density=True)
    axes[1, i].set_title(f"{sname}\nunknown"); axes[1, i].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_ROOT}/figures/fig07_distribution.png"); plt.close()

# -----------------------------------------------------------------------------
# SECTION 11 — ADVERSARIAL PERTURBATIONS
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 11 — ADVERSARIAL PERTURBATION EVALUATION")
print("=" * 80)

def perturb_timestamp(text, rng_a):
    return re.sub(r"\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}",
                  lambda _: f"2026-{rng_a.randint(1,13):02d}-{rng_a.randint(1,29):02d}T"
                            f"{rng_a.randint(0,24):02d}:{rng_a.randint(0,60):02d}:"
                            f"{rng_a.randint(0,60):02d}", text)
def perturb_field_order(text, rng_a):
    parts = text.split(" | "); rng_a.shuffle(parts); return " | ".join(parts)
def perturb_numeric(text, rng_a):
    def repl(m):
        v = float(m.group(0)); return f"{v * (1 + rng_a.uniform(-0.2, 0.2)):.6f}"
    return re.sub(r"\d+\.\d+", repl, text)
def perturb_irrelevant_field(text, rng_a):
    return text + f" | aux_field_{rng_a.randint(0,1000)}: {rng_a.randint(0,1000)}"

PERTURBATIONS = {
    "timestamp": perturb_timestamp,
    "field_order": perturb_field_order,
    "numeric": perturb_numeric,
    "irrelevant_field": perturb_irrelevant_field,
}

rng_a = np.random.RandomState(0)
adv_rows = []
for pname, pfn in PERTURBATIONS.items():
    unk_texts = [TEST_TEXTS[i] for i in np.where(um)[0]]
    n = min(500, len(unk_texts))
    idxs = rng_a.choice(len(unk_texts), size=n, replace=False)
    perturbed = [pfn(unk_texts[i], rng_a) for i in idxs]
    E_p = embedder_ft.get_embeddings(perturbed, use_cache=False)
    S_p = np.array([retr_ft.query(e) for e in E_p])
    flags = ((S_p[:, 0] < tp_ours) | (S_p[:, 2] > tu_ours)).astype(float)
    adv_rows.append({"perturbation": pname, "n": n,
                     "unknown_recall": float(flags.mean()),
                     "FAR": 1 - float(flags.mean())})

df_adv = pd.DataFrame(adv_rows)
print(df_adv.to_string(index=False))
df_adv.to_csv(f"{OUT_ROOT}/adversarial/perturbation_results.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(df_adv['perturbation'], df_adv['unknown_recall'])
ax.set_ylim(0, 1.05); ax.set_ylabel('Unknown recall after perturbation')
ax.set_title('Adversarial perturbation robustness (500 samples each)')
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/figures/fig08_adversarial.png"); plt.close()

# -----------------------------------------------------------------------------
# SECTION 12 — CROSS-FAMILY GENERALIZATION
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 12 — CROSS-FAMILY GENERALIZATION")
print("=" * 80)

cross_rows = []
for f_known in KNOWN_ATTACKS:
    tr = df_train[df_train['Attack Type'] == f_known]
    if len(tr) < 5: continue
    E_tr = embedder_ft.get_embeddings(tr['Log'].tolist())
    y_tr = [f_known] * len(tr)
    protos = build_prototypes(E_tr, y_tr)
    pc = PrototypeClassifier(protos)
    idx = build_faiss_index(E_tr)
    retr = MultiSignalRetriever(idx, E_tr, y_tr, pc, k=min(30, len(E_tr)))
    for g_unknown in KNOWN_ATTACKS:
        if g_unknown == f_known: continue
        te = df_train[df_train['Attack Type'] == g_unknown]
        if len(te) < 5: continue
        te = te.sample(n=min(50, len(te)), random_state=0)
        E_te = embedder_ft.get_embeddings(te['Log'].tolist())
        S = np.array([retr.query(e) for e in E_te])
        rej = (S[:, 0] < 0.5).astype(float)
        cross_rows.append({"known": f_known, "unknown": g_unknown,
                           "rejection": float(rej.mean()), "n": len(te)})

df_cross = pd.DataFrame(cross_rows)
print(df_cross.to_string(index=False))
df_cross.to_csv(f"{OUT_ROOT}/cross_family/cross_family.csv", index=False)

if len(df_cross):
    pivot = df_cross.pivot(index="known", columns="unknown", values="rejection")
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
    ax.set_title('Cross-family rejection (single known -> all other as unknown)')
    plt.tight_layout(); plt.savefig(f"{OUT_ROOT}/figures/fig09_cross_family.png"); plt.close()

# -----------------------------------------------------------------------------
# SECTION 13 — FINAL SUMMARY + SAVE
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("FINAL SUMMARY — HONEST CLAIMS")
print("=" * 80)

final = {
    "protocol_A_mean_rejection": float(df_A['rejection_rate'].mean()) if len(df_A) else float('nan'),
    "protocol_B_mean_rejection": float(df_B['rejection_rate'].mean()) if len(df_B) else float('nan'),
    "leakage_gap": float(leak_gap),
    "fixed_test_set_size": len(TEST_TEXTS),
    "fixed_test_known": int(km.sum()),
    "fixed_test_unknown": int(um.sum()),
    "fixed_test_benign": int(bm.sum()),
    "fusion_table": df_fusion.to_dict(orient="records"),
    "signal_ablation": df_abl.to_dict(orient="records"),
    "calibration_ablation": df_cal.to_dict(orient="records"),
    "seeds_mean_std": {c: {"mean": float(df_seeds[c].mean()),
                           "std": float(df_seeds[c].std())}
                       for c in ["known_acc","unknown_recall","FAR","FRR","benign_rejection"]},
    "family_bootstrap_recall": {
        "mean": float(boot_recs.mean()),
        "ci95_low": float(np.percentile(boot_recs, 2.5)),
        "ci95_high": float(np.percentile(boot_recs, 97.5)),
    },
    "scaling": df_scaling.to_dict(orient="records"),
    "distribution": df_dist.to_dict(orient="records"),
    "adversarial": df_adv.to_dict(orient="records"),
    "cross_family_mean_rejection": float(df_cross['rejection'].mean()) if len(df_cross) else float('nan'),
    "primary_operating_point": {"tau_p": float(tp_ours), "tau_u": float(tu_ours)},
}

with open(f"{OUT_ROOT}/final_summary.json", "w") as f:
    json.dump(final, f, indent=2, default=str)

print(f"  Protocol A mean rejection (leaky)  : {final['protocol_A_mean_rejection']:.4f}")
print(f"  Protocol B mean rejection (strict) : {final['protocol_B_mean_rejection']:.4f}")
print(f"  Leakage gap                        : {final['leakage_gap']:+.4f}")
print(f"  Family-level bootstrap recall CI   : "
      f"[{final['family_bootstrap_recall']['ci95_low']:.4f}, "
      f"{final['family_bootstrap_recall']['ci95_high']:.4f}]")
print(f"  Cross-family mean rejection        : {final['cross_family_mean_rejection']:.4f}")
print(f"  Adversarial recall range           : "
      f"[{df_adv['unknown_recall'].min():.4f}, {df_adv['unknown_recall'].max():.4f}]")
print(f"  Primary operating point            : "
      f"tau_p={tp_ours:.4f}  tau_u={tu_ours:.4f}")

zip_path = f"{OUT_ROOT}/codaspy_audit_bundle.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, fs in os.walk(OUT_ROOT):
        for fn in fs:
            if fn.endswith(".zip"): continue
            z.write(os.path.join(root, fn),
                    arcname=os.path.relpath(os.path.join(root, fn), OUT_ROOT))
print(f"\n[saved] {zip_path}")

if _HAVE_COLAB:
    try:
        _colab_files.download(zip_path)
    except Exception as e:
        print(f"[note] download skipped: {e}")

print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

CODASPY 2027 AUDIT + REPAIR (v3, corrected embedder)
Device: cuda
GPU: Tesla T4
Seeds: [42, 1, 7, 13, 23]

SECTION 1 — DATASETS (KNOWN / UNKNOWN_ATTACK / BENIGN)
Training rows : 4500
Sample classes: {'UNKNOWN_ATTACK': np.int64(607), 'BENIGN': np.int64(100), 'KNOWN': np.int64(100)}
Unknown rows  : 607
Benign rows   : 100

FIXED TEST SET: 750
  known   : 450
  unknown : 200
  benign  : 100

SECTION 2 — ENCODERS
[finetuned] loaded (skipped 12 head weights)


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[base] loaded

SECTION 3 — EMBEDDING, PROTOTYPES, MULTI-SIGNAL RETRIEVER
[CHECK] embedder returns full arrays: (200, 768)

SECTION 4 — LOFO PROTOCOLS

Running Protocol A (encoder-leaky)...
  [A] holding out DNS Fast-Flux
     tau_p=0.9640 tau_u=0.0080 pseudo_FRR=1.0000 pseudo_rec=0.0000 | pk=4672 pu=640
  [A] holding out DoS
     tau_p=0.9663 tau_u=0.0080 pseudo_FRR=1.0000 pseudo_rec=0.0000 | pk=4672 pu=640
  [A] holding out DoS + Brute-Force
     tau_p=0.9631 tau_u=0.0085 pseudo_FRR=1.0000 pseudo_rec=0.0000 | pk=4672 pu=640
  [A] holding out FTP Brute-Force / Data Exfiltration
     tau_p=0.9632 tau_u=0.0085 pseudo_FRR=1.0000 pseudo_rec=0.0000 | pk=4672 pu=640
  [A] holding out HTTP C2
     tau_p=0.9678 tau_u=0.0076 pseudo_FRR=1.0000 pseudo_rec=0.0000 | pk=4672 pu=640
  [A] holding out ICMP Flood
     tau_p=0.9662 tau_u=0.0079 pseudo_FRR=1.0000 pseudo_rec=0.0000 | pk=4672 pu=640
  [A] holding out IRC C2
     tau_p=0.9654 tau_u=0.0082 pseudo_FRR=1.0000 pseudo_rec=0.0000 | pk=4672 pu=640

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

     tau_p=0.9242 tau_u=0.0188 pseudo_FRR=1.0000 pseudo_rec=0.0000 | pk=4672 pu=640

Protocol A — held-out rejection (leaky encoder)
                             family  n_held  rejection_rate    tau_p    tau_u  pseudo_frr  pseudo_rec
                      DNS Fast-Flux     200           1.000 0.964034 0.007973         1.0         0.0
                                DoS     200           0.975 0.966328 0.007973         1.0         0.0
                  DoS + Brute-Force     200           1.000 0.963069 0.008513         1.0         0.0
FTP Brute-Force / Data Exfiltration     200           0.985 0.963164 0.008513         1.0         0.0
                            HTTP C2     200           0.975 0.967819 0.007620         1.0         0.0
                         ICMP Flood     200           0.800 0.966190 0.007946         1.0         0.0
                             IRC C2     200           1.000 0.965388 0.008206         1.0         0.0
                     P2P / UDP Scan     200        

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


AUDIT COMPLETE
